In [ ]:
# ============================================================
# MOD-09 — Modelado Predictivo de Rendimiento Arrocero
# Proyecto: Impacto Variabilidad Climatica en Produccion Arrocera
# Version  : 3.0 — PCA para seleccion de componentes predictivos
# Entrada  : data/master/A3_DF_Features.parquet   (MOD-08)
# Salidas  : data/master/M1_resultados_modelos.xlsx
#            data/master/M1_modelo_final.pkl
#            data/master/M1_PCA_componentes.pkl
# ============================================================
# Modelos entrenados y contrastados:
#   0. Ridge Regression        — baseline lineal
#   1. Random Forest           — modelo principal recomendado
#   2. XGBoost                 — contraste boosting
#   3. LightGBM                — contraste boosting eficiente
#
# Estrategia de validacion:
#   Leave-One-Period-Out (LOPO) — respeta estructura temporal
#
# Seleccion de variables v3.0 — PCA:
#   Se reemplaza la seleccion Top-30 por SHAP con PCA.
#   PCA identifica las combinaciones lineales de features que
#   explican la mayor varianza conjunta del espacio climatico.
#   Los componentes seleccionados son los que aportan mas
#   informacion al proceso de prediccion, cuantificado por:
#     a) Varianza explicada acumulada (criterio estadistico)
#     b) Correlacion de cada componente con el target LOPO
#     c) Carga de cada feature original en cada componente
# ============================================================
!pip install scikit-learn xgboost lightgbm shap openpyxl pyarrow -q

import os, gc, time, warnings, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.ticker as mticker
from matplotlib.colors import LinearSegmentedColormap

from sklearn.linear_model    import Ridge
from sklearn.ensemble        import RandomForestRegressor
from sklearn.preprocessing   import LabelEncoder, StandardScaler
from sklearn.decomposition   import PCA
from sklearn.metrics         import (mean_squared_error, mean_absolute_error,
                                     r2_score, mean_absolute_percentage_error,
                                     confusion_matrix, roc_curve, auc)
from sklearn.multiclass      import OneVsRestClassifier
from sklearn.preprocessing   import label_binarize
from sklearn.model_selection import ParameterGrid

from xgboost  import XGBRegressor
from lightgbm import LGBMRegressor
import shap

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 110
print('OK Librerias cargadas correctamente')
print('   sklearn.decomposition.PCA incluido')


In [ ]:
# ============================================================
# CONFIGURACIÓN DE RUTAS Y PARÁMETROS — MOD-09
# ============================================================

from google.colab import drive
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
    print('✅ Google Drive montado')
else:
    print('✅ Google Drive ya montado')

# ── Detección automática ─────────────────────────────────────
RUTA_BASE = None
print()
print('🔍 Buscando carpeta del proyecto...')
for dirpath, dirnames, filenames in os.walk('/content/drive/MyDrive'):
    depth = dirpath.replace('/content/drive/MyDrive','').count(os.sep)
    if depth > 5:
        dirnames.clear(); continue
    if 'A3_DF_Features.parquet' in filenames:
        candidate = os.path.dirname(os.path.dirname(dirpath))
        if os.path.isdir(os.path.join(candidate,'data','master')):
            RUTA_BASE = candidate
            print(f'   ✅ Proyecto en: {RUTA_BASE}'); break

if RUTA_BASE is None:
    RUTA_BASE = '/content/drive/MyDrive/Colab Notebooks/Proyecto_Aplicado_II'
    print(f'   ⚠️  Usando ruta manual: {RUTA_BASE}')

RUTA_MASTER = os.path.join(RUTA_BASE, 'data', 'master')
os.makedirs(RUTA_MASTER, exist_ok=True)

# ── Archivos ──────────────────────────────────────────────────
ENTRADA_FEATURES    = os.path.join(RUTA_MASTER, 'A3_DF_Features.parquet')
SALIDA_RESULTADOS   = os.path.join(RUTA_MASTER, 'M1_resultados_modelos.xlsx')
SALIDA_MODELO_PKL   = os.path.join(RUTA_MASTER, 'M1_modelo_final.pkl')
SALIDA_SHAP_PKL     = os.path.join(RUTA_MASTER, 'M2_shap_values.pkl')
SALIDA_DASHBOARD    = os.path.join(RUTA_MASTER, 'M1_Dashboard_Modelos.png')
SALIDA_RF_OPT_PKL   = os.path.join(RUTA_MASTER, 'M1_RF_Optimizado.pkl')
SALIDA_RF_TOP30_PKL = os.path.join(RUTA_MASTER, 'M1_RF_Top30.pkl')
SALIDA_RF_TOP30_XLS = os.path.join(RUTA_MASTER, 'M1_RF_PCA_Resultados.xlsx')
SALIDA_PCA_PKL      = os.path.join(RUTA_MASTER, 'M1_PCA_componentes.pkl')
SALIDA_RF_TOP30_PKL = os.path.join(RUTA_MASTER, 'M1_RF_PCA.pkl')


# ── Parámetros de modelado ────────────────────────────────────
TARGET_COL       = 'Rendimiento (t/ha)'  # variable objetivo continua
SEMILLA          = 42                    # reproducibilidad
N_CLASES         = 3                     # Bajo / Medio / Alto
CUANTILES        = [0.33, 0.66]          # cortes para clasificación
NOMBRES_CLASES   = ['Bajo', 'Medio', 'Alto']
COLS_EXCLUIR_X   = [                     # no usar como features
    'Periodo','Departamento','Municipio','Año','Semestre',
    'Cultivo','Desagregación cultivo','Es_Top500','Cuadrante',
    'Producción (t)','Área (ha)',        # targets alternativos
    'Rendimiento (t/ha)',                # target principal
]

# ── Identificadores geográficos a excluir ──────────────────
# DECISIÓN v2: 'Código Dane municipio' fue detectado como feature
# top-5 en SHAP del MOD-09 v1. Su presencia implica que el modelo
# memoriza el histórico por municipio (efecto fijo geográfico)
# en lugar de aprender la relación clima → rendimiento.
# Se excluye para forzar generalización basada en variables climáticas.
COLS_EXCLUIR_IDENTIF = [
    'Código Dane municipio',
    'codigo_dane',
    'codigo_municipio',
    'DANE',
]
# PCA v3.0 — criterios de seleccion de componentes
PCA_VARIANZA_OBJETIVO  = 0.90   # componentes que explican >= 90% varianza acumulada
PCA_VARIANZA_MIN_COMP  = 0.01   # descartar componentes que aporten < 1% varianza
PCA_MAX_COMPONENTES    = 30     # limite maximo de componentes (evitar sobredimension)
TOP_N_FEATURES = 30             # mantenido por compatibilidad con celda 15


# ── Hiperparámetros de cada modelo ────────────────────────────
PARAMS_RF = dict(
    n_estimators = 300,
    max_features = 'sqrt',
    max_depth    = None,
    min_samples_leaf = 3,
    random_state = SEMILLA,
    n_jobs       = -1
)
PARAMS_XGB = dict(
    n_estimators      = 400,
    learning_rate     = 0.05,
    max_depth         = 4,
    subsample         = 0.8,
    colsample_bytree  = 0.8,
    reg_alpha         = 0.1,
    reg_lambda        = 1.0,
    early_stopping_rounds = 20,
    eval_metric       = 'rmse',
    random_state      = SEMILLA,
    verbosity         = 0,
    n_jobs            = -1
)
PARAMS_LGB = dict(
    n_estimators      = 400,
    learning_rate     = 0.05,
    num_leaves        = 15,
    max_depth         = 4,
    subsample         = 0.8,
    colsample_bytree  = 0.8,
    reg_alpha         = 0.1,
    reg_lambda        = 1.0,
    min_child_samples = 5,
    random_state      = SEMILLA,
    n_jobs            = -1,
    verbose           = -1
)
PARAMS_RIDGE = dict(alpha=10.0)

if not os.path.exists(ENTRADA_FEATURES):
    raise FileNotFoundError(
        f'No se encontró A3_DF_Features.parquet.\nEjecuta MOD-08 antes.')

print()
print('📂 Rutas configuradas:')
print(f'   Master    → {RUTA_MASTER}')
print(f'   Entrada   → A3_DF_Features.parquet  ✅')
print()
print('⚙️  Parámetros:')
print(f'   Target        : {TARGET_COL}')
print(f'   N clases (ROC): {N_CLASES}  ({", ".join(NOMBRES_CLASES)})')
print(f'   Semilla       : {SEMILLA}')


In [ ]:
# ============================================================
# FUNCIONES UTILITARIAS — MOD-09
# ============================================================

def calcular_metricas_regresion(y_true, y_pred, nombre=''):
    """Calcula RMSE, MAE, R², MAPE sobre arrays reales vs predichos."""
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred) * 100
    if nombre:
        print(f'   {nombre:<20} RMSE={rmse:.4f}  MAE={mae:.4f}  '
              f'R²={r2:.4f}  MAPE={mape:.2f}%')
    return {'RMSE': rmse, 'MAE': mae, 'R2': r2, 'MAPE': mape}


def rendimiento_a_clases(y_series, cuantiles=CUANTILES):
    """
    Convierte rendimiento continuo (t/ha) a clases ordinales.
    Cortes por cuantiles 33/66 para balancear clases.
    Retorna (y_clases_int, cortes, nombres_clases).
    """
    corte_bajo, corte_alto = y_series.quantile(cuantiles[0]), y_series.quantile(cuantiles[1])
    clases = pd.cut(
        y_series,
        bins   = [-np.inf, corte_bajo, corte_alto, np.inf],
        labels = [0, 1, 2]
    ).astype(int)
    return clases, (corte_bajo, corte_alto)


def lopo_cv(modelo_cls, X, y, periodos, params_extra=None,
            nombre='Modelo', is_xgb=False, is_lgb=False):
    """
    Leave-One-Period-Out Cross-Validation.

    JUSTIFICACIÓN DE LOPO sobre K-Fold estándar:
    ─────────────────────────────────────────────
    El dataset tiene estructura de panel temporal:
    (municipio × periodo). Un K-Fold aleatorio mezclaría
    periodos de entrenamiento y test, filtrando información
    futura al modelo (data leakage temporal).
    LOPO excluye un periodo completo en cada fold y entrena
    con todos los demás, simulando el escenario real de uso:
    predecir el rendimiento de un semestre futuro dado el
    historial de semestres anteriores y el mismo periodo
    en otros municipios.
    Limitación: con 10 periodos solo hay 10 folds → alta
    varianza en las métricas, subsanada promediando y
    reportando desv. estándar.

    Retorna: lista de dicts con métricas por fold.
    """
    periodos_unicos = sorted(periodos.unique())
    resultados = []

    for per_test in periodos_unicos:
        mask_test  = (periodos == per_test)
        mask_train = ~mask_test

        X_tr, X_te = X[mask_train], X[mask_test]
        y_tr, y_te = y[mask_train], y[mask_test]

        if len(y_te) == 0:
            continue

        # Escalado para Ridge (no necesario para árboles)
        if not (is_xgb or is_lgb) and hasattr(modelo_cls, 'coef_') is False:
            scaler = StandardScaler()
            X_tr_f = scaler.fit_transform(X_tr)
            X_te_f = scaler.transform(X_te)
        else:
            X_tr_f, X_te_f = X_tr, X_te

        # Clonar instancia para no acumular estado
        import copy
        modelo = copy.deepcopy(modelo_cls)

        try:
            if is_xgb:
                modelo.fit(X_tr_f, y_tr,
                           eval_set=[(X_te_f, y_te)],
                           verbose=False)
            elif is_lgb:
                from lightgbm import early_stopping, log_evaluation
                callbacks = [early_stopping(20, verbose=False),
                             log_evaluation(period=-1)]
                modelo.fit(X_tr_f, y_tr,
                           eval_set=[(X_te_f, y_te)],
                           callbacks=callbacks)
            else:
                modelo.fit(X_tr_f, y_tr)

            y_pred = modelo.predict(X_te_f)
            metricas = calcular_metricas_regresion(y_te, y_pred)
            metricas['Periodo_test'] = per_test
            metricas['n_test']       = len(y_te)
            metricas['n_train']      = len(y_tr)
            resultados.append(metricas)
        except Exception as e:
            print(f'     Fold {per_test}: error — {e}')

    return resultados


def resumir_lopo(resultados, nombre):
    """Calcula media ± std de métricas LOPO."""
    df_r = pd.DataFrame(resultados)
    resumen = {}
    for m in ['RMSE','MAE','R2','MAPE']:
        if m in df_r.columns:
            resumen[f'{m}_mean'] = df_r[m].mean()
            resumen[f'{m}_std']  = df_r[m].std()
    resumen['Modelo']   = nombre
    resumen['n_folds']  = len(df_r)
    print(f'   {nombre:<22}: '
          f'RMSE={resumen["RMSE_mean"]:.4f}±{resumen["RMSE_std"]:.4f}  '
          f'MAE={resumen["MAE_mean"]:.4f}  '
          f'R²={resumen["R2_mean"]:.4f}±{resumen["R2_std"]:.4f}  '
          f'MAPE={resumen["MAPE_mean"]:.2f}%')
    return resumen, df_r


print('✅ Funciones utilitarias cargadas')
print()
print('Validación:  Leave-One-Period-Out (LOPO)')
print('Conversión a clases: cuantiles 33/66 → Bajo / Medio / Alto')


In [ ]:
# ============================================================
# CARGAR A3_DF_Features Y PREPARAR MATRICES X, y
# ============================================================

print('📥 Cargando A3_DF_Features.parquet...')
df_feat = pd.read_parquet(ENTRADA_FEATURES)

for col in ['Periodo','Departamento','Municipio']:
    if col in df_feat.columns:
        df_feat[col] = df_feat[col].astype(str).str.strip().str.upper()

print(f'   Shape cargado : {df_feat.shape[0]:,} × {df_feat.shape[1]}')
print(f'   Periodos      : {sorted(df_feat["Periodo"].unique())}')
print(f'   Departamentos : {df_feat["Departamento"].nunique()}')
print(f'   Municipios    : {df_feat["Municipio"].nunique()}')

# ── Verificar target ──────────────────────────────────────────
if TARGET_COL not in df_feat.columns:
    cols_posibles = [c for c in df_feat.columns if 'endimiento' in c]
    if cols_posibles:
        TARGET_COL = cols_posibles[0]
        print(f'   ⚠️  TARGET_COL ajustado a: {TARGET_COL}')
    else:
        raise KeyError(f'No se encontró columna de rendimiento. Disponibles: {df_feat.columns.tolist()}')

# ── Eliminar filas sin target ─────────────────────────────────
n_antes = len(df_feat)
df_model = df_feat[df_feat[TARGET_COL].notna()].copy().reset_index(drop=True)
print(f'\n   Filas con target válido: {len(df_model):,} / {n_antes:,}')

# ── Construir X e y ───────────────────────────────────────────
# ── EXCLUIR identificadores geográficos (v2) ─────────────────
# Razón: 'Código Dane municipio' apareció top-5 en SHAP v1,
# indicando memorización por municipio. Se excluye para que el
# modelo aprenda relación clima→rendimiento, no efecto fijo.
cols_x = [c for c in df_model.columns
          if c not in COLS_EXCLUIR_X and
          c not in ['Área cosechada (ha)','Área sembrada (ha)',
                    'Producción (t)'] and
          # Excluir identificadores numéricos geográficos
          not any(exc.lower() in c.lower() for exc in COLS_EXCLUIR_IDENTIF) and
          df_model[c].dtype in ['float32','float64','int64','Int64','int32']]

print(f'   Columnas excluidas por identificador geográfico:')
for exc in COLS_EXCLUIR_IDENTIF:
    cols_match = [c for c in df_model.columns if exc.lower() in c.lower()]
    if cols_match:
        print(f'     ✂️  {cols_match}')

X = df_model[cols_x].copy()
y = df_model[TARGET_COL].copy()

# Rellenar nulos residuales con mediana
X = X.fillna(X.median(numeric_only=True))

# Convertir Int64 (nullable) a float32
for c in X.columns:
    if str(X[c].dtype).startswith('Int'):
        X[c] = X[c].astype('float32')

PERIODOS_SERIE = df_model['Periodo']

print(f'\n   Features (X) : {X.shape[1]} columnas')
print(f'   Target  (y)  : {y.name}')
print(f'   Rango target : {y.min():.2f} – {y.max():.2f} t/ha')
print(f'   Media target : {y.mean():.2f} t/ha  |  Mediana: {y.median():.2f} t/ha')
print(f'   Desv. estándar: {y.std():.2f} t/ha')
print()

# ── Distribución de clases (para clasificación) ───────────────
y_clases, (corte_bajo, corte_alto) = rendimiento_a_clases(y)
print(f'   Cortes para clases (cuantiles 33/66):')
print(f'   Bajo   : rendimiento <  {corte_bajo:.2f} t/ha')
print(f'   Medio  : {corte_bajo:.2f} – {corte_alto:.2f} t/ha')
print(f'   Alto   : rendimiento >  {corte_alto:.2f} t/ha')
print()
for i, nombre_c in enumerate(NOMBRES_CLASES):
    n_c = (y_clases == i).sum()
    print(f'   Clase {nombre_c:<6}: {n_c:>4} obs ({n_c/len(y)*100:.1f}%)')

# Diagnóstico: nulos en X
nulos_x = X.isna().sum().sum()
print(f'\n   Nulos en X post-limpieza: {nulos_x}')
if nulos_x > 0:
    print(f'   ⚠️  Columnas con nulos: {X.columns[X.isna().any()].tolist()}')


In [ ]:
# ============================================================
# VALIDACIÓN LOPO Y ENTRENAMIENTO FINAL — 4 modelos
# ============================================================

sep = '='*65
print(sep)
print('  VALIDACIÓN LEAVE-ONE-PERIOD-OUT (LOPO)')
print(sep)
print()
print(f'  Folds     : {PERIODOS_SERIE.nunique()} (uno por periodo)')
print(f'  Estrategia: excluir 1 periodo como test, entrenar con resto')
print(f'  Métricas  : RMSE (principal) · MAE · R² · MAPE')
print()

X_arr = X.values.astype(float)
y_arr = y.values.astype(float)

# ── 0. Ridge Regression (baseline) ───────────────────────────
print('0. Ridge Regression (baseline lineal):')
scaler_ridge = StandardScaler()
X_scaled = scaler_ridge.fit_transform(X_arr)
ridge_base = Ridge(**PARAMS_RIDGE)
lopo_ridge = lopo_cv(ridge_base, X_scaled, y_arr, PERIODOS_SERIE,
                     nombre='Ridge')
res_ridge, df_lopo_ridge = resumir_lopo(lopo_ridge, 'Ridge')

# ── 1. Random Forest ─────────────────────────────────────────
print()
print('1. Random Forest:')
rf_base = RandomForestRegressor(**PARAMS_RF)
lopo_rf = lopo_cv(rf_base, X_arr, y_arr, PERIODOS_SERIE,
                  nombre='Random Forest')
res_rf, df_lopo_rf = resumir_lopo(lopo_rf, 'Random Forest')

# ── 2. XGBoost ───────────────────────────────────────────────
print()
print('2. XGBoost:')
xgb_base = XGBRegressor(**PARAMS_XGB)
lopo_xgb = lopo_cv(xgb_base, X_arr, y_arr, PERIODOS_SERIE,
                   nombre='XGBoost', is_xgb=True)
res_xgb, df_lopo_xgb = resumir_lopo(lopo_xgb, 'XGBoost')

# ── 3. LightGBM ──────────────────────────────────────────────
print()
print('3. LightGBM:')
lgb_base = LGBMRegressor(**PARAMS_LGB)
lopo_lgb = lopo_cv(lgb_base, X_arr, y_arr, PERIODOS_SERIE,
                   nombre='LightGBM', is_lgb=True)
res_lgb, df_lopo_lgb = resumir_lopo(lopo_lgb, 'LightGBM')

# ── Tabla comparativa de resultados LOPO ─────────────────────
print()
print(sep)
print('  TABLA COMPARATIVA — Validación LOPO')
print(sep)
RESUMEN_LOPO = pd.DataFrame([res_ridge, res_rf, res_xgb, res_lgb])
cols_show = ['Modelo','RMSE_mean','RMSE_std','MAE_mean','R2_mean','R2_std','MAPE_mean']
print()
print(RESUMEN_LOPO[cols_show].to_string(index=False, float_format=lambda x: f'{x:.4f}'))

mejor_modelo_lopo = RESUMEN_LOPO.loc[RESUMEN_LOPO['RMSE_mean'].idxmin(), 'Modelo']
print()
print(f'  Modelo con menor RMSE_LOPO: {mejor_modelo_lopo}')


In [ ]:
# ============================================================
# ENTRENAMIENTO FINAL — Modelos sobre el dataset completo
# (para SHAP, matrices de confusión y curvas ROC)
# ============================================================

print('Entrenando modelos finales sobre dataset completo...')
print()

# Ridge final
scaler_final = StandardScaler()
X_scaled_final = scaler_final.fit_transform(X_arr)
ridge_final = Ridge(**PARAMS_RIDGE)
ridge_final.fit(X_scaled_final, y_arr)
y_pred_ridge = ridge_final.predict(X_scaled_final)
m_ridge_full = calcular_metricas_regresion(y_arr, y_pred_ridge, 'Ridge  (full)')

# RF final
rf_final = RandomForestRegressor(**PARAMS_RF)
rf_final.fit(X_arr, y_arr)
y_pred_rf = rf_final.predict(X_arr)
m_rf_full = calcular_metricas_regresion(y_arr, y_pred_rf, 'RF     (full)')

# XGB final (sin early stopping)
params_xgb_full = {k:v for k,v in PARAMS_XGB.items()
                   if k != 'early_stopping_rounds'}
params_xgb_full['n_estimators'] = 300
xgb_final = XGBRegressor(**params_xgb_full)
xgb_final.fit(X_arr, y_arr)
y_pred_xgb = xgb_final.predict(X_arr)
m_xgb_full = calcular_metricas_regresion(y_arr, y_pred_xgb, 'XGBoost(full)')

# LGB final
lgb_final = LGBMRegressor(**{k:v for k,v in PARAMS_LGB.items()})
lgb_final.fit(X_arr, y_arr)
y_pred_lgb = lgb_final.predict(X_arr)
m_lgb_full = calcular_metricas_regresion(y_arr, y_pred_lgb, 'LightGBM(full)')

print()
print('NOTA: Las métricas sobre dataset completo miden el ajuste en')
print('muestra (in-sample). Las métricas LOPO son las relevantes para')
print('evaluar capacidad de generalización (fuera de muestra).')

# ── Convertir predicciones a clases ──────────────────────────
def pred_a_clases(y_pred_cont, corte_bajo, corte_alto):
    clases = np.zeros(len(y_pred_cont), dtype=int)
    clases[y_pred_cont >= corte_bajo]  = 1
    clases[y_pred_cont >= corte_alto]  = 2
    return clases

Y_CLASE_REAL  = y_clases.values
Y_PRED_RIDGE_C = pred_a_clases(y_pred_ridge, corte_bajo, corte_alto)
Y_PRED_RF_C    = pred_a_clases(y_pred_rf,    corte_bajo, corte_alto)
Y_PRED_XGB_C   = pred_a_clases(y_pred_xgb,  corte_bajo, corte_alto)
Y_PRED_LGB_C   = pred_a_clases(y_pred_lgb,  corte_bajo, corte_alto)

print()
print('Predicciones convertidas a clases (Bajo/Medio/Alto).')


In [ ]:
# ============================================================
# MATRICES DE CONFUSIÓN — 4 modelos
#
# INTERPRETACIÓN DE LA MATRIZ DE CONFUSIÓN:
# ──────────────────────────────────────────
# Filas   = Clase real (Bajo / Medio / Alto)
# Columnas= Clase predicha
# Diagonal principal (verde) = predicciones correctas
# Fuera de diagonal = errores de clasificación
#
# Para un modelo de regresión convertido a clases:
#   - Los errores adyacentes (Bajo→Medio, Medio→Alto) son
#     tolerables: el modelo "casi acertó" en magnitud.
#   - Los errores distantes (Bajo→Alto, Alto→Bajo) son
#     críticos: el modelo invirtió la dirección del error.
#
# ADVERTENCIA METODOLÓGICA:
#   Estas matrices se calculan sobre el dataset COMPLETO
#   (in-sample). Los patrones de error deben interpretarse
#   junto con los resultados LOPO para evitar conclusiones
#   optimistas. Un modelo que memoriza el dataset tendrá
#   una diagonal perfecta aquí pero RMSE_LOPO alto.
# ============================================================

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
fig.suptitle(
    'Matrices de Confusión — Rendimiento en 3 Clases (Bajo / Medio / Alto)\n'
    'Evaluación in-sample | Clases por cuantiles 33/66',
    fontsize=12, fontweight='bold'
)

cmap_matriz = LinearSegmentedColormap.from_list(
    'custom', ['#FFFFFF', '#1565C0'], N=256)

modelos_conf = [
    ('Ridge (baseline)',   Y_PRED_RIDGE_C),
    ('Random Forest',      Y_PRED_RF_C),
    ('XGBoost',            Y_PRED_XGB_C),
    ('LightGBM',           Y_PRED_LGB_C),
]

for ax, (nombre, y_pred_c) in zip(axes, modelos_conf):
    cm = confusion_matrix(Y_CLASE_REAL, y_pred_c, labels=[0,1,2])
    acc_total  = np.trace(cm) / cm.sum()
    err_adyac  = (cm[0,1]+cm[1,0]+cm[1,2]+cm[2,1]) / cm.sum()
    err_distal = (cm[0,2]+cm[2,0]) / cm.sum()

    im = ax.imshow(cm, cmap=cmap_matriz, aspect='auto')
    ax.set_xticks([0,1,2]); ax.set_yticks([0,1,2])
    ax.set_xticklabels(NOMBRES_CLASES, fontsize=9)
    ax.set_yticklabels(NOMBRES_CLASES, fontsize=9)
    ax.set_xlabel('Predicho', fontsize=9)
    ax.set_ylabel('Real', fontsize=9)
    ax.set_title(
        f'{nombre}\nAcc={acc_total:.2%}  Err_adj={err_adyac:.2%}  Err_dist={err_distal:.2%}',
        fontsize=9, fontweight='bold'
    )
    max_val = cm.max()
    for i in range(3):
        for j in range(3):
            color = 'white' if cm[i,j] > max_val*0.6 else 'black'
            ax.text(j, i, str(cm[i,j]), ha='center', va='center',
                    color=color, fontsize=12, fontweight='bold')

plt.colorbar(im, ax=axes[-1], fraction=0.046, pad=0.04)
plt.tight_layout()
plt.savefig(os.path.join(RUTA_MASTER, 'M1_Matrices_Confusion.png'),
            dpi=130, bbox_inches='tight')
plt.show()
print('✅ Matrices de confusión guardadas: M1_Matrices_Confusion.png')
print()

# Imprimir tabla de métricas de clasificación
print('Resumen de errores in-sample (sobre dataset completo):')
print(f'{"Modelo":<22}  {"Acc":>6}  {"Err_adj":>8}  {"Err_dist":>9}  Interpretación')
print('-'*75)
for nombre, y_pred_c in modelos_conf:
    cm = confusion_matrix(Y_CLASE_REAL, y_pred_c, labels=[0,1,2])
    acc  = np.trace(cm) / cm.sum()
    adj  = (cm[0,1]+cm[1,0]+cm[1,2]+cm[2,1]) / cm.sum()
    dist = (cm[0,2]+cm[2,0]) / cm.sum()
    interp = 'Memoriza (overfit)' if acc > 0.95 else (
             'Buena disc.' if acc > 0.75 else 'Moderada')
    print(f'  {nombre:<22}  {acc:>5.1%}  {adj:>7.1%}  {dist:>8.1%}  {interp}')


In [ ]:
# ============================================================
# CURVAS ROC MULTICLASE — One-vs-Rest por modelo
#
# INTERPRETACIÓN DE LAS CURVAS ROC:
# ──────────────────────────────────
# La curva ROC grafica TPR (Recall) vs FPR para distintos
# umbrales de clasificación. El área bajo la curva (AUC)
# resume la capacidad discriminativa: 1.0 = perfecto,
# 0.5 = aleatorio.
#
# Para modelos de REGRESIÓN convertidos a clases:
#   Se usa el valor continuo predicho como "score" de
#   probabilidad implícita para cada clase (One-vs-Rest).
#   Un modelo con AUC > 0.80 distingue bien las clases.
#   Un AUC cercano a 1.0 in-sample en un modelo de árbol
#   profundo señala sobreajuste, no excelencia real.
#
# ADVERTENCIA:
#   AUC in-sample de RF/XGB/LGB > 0.95 es esperado y NO
#   implica que el modelo generalice bien. El AUC relevante
#   sería el calculado sobre los folds LOPO, que se estima
#   indirectamente a través del RMSE de validación.
# ============================================================

y_bin = label_binarize(Y_CLASE_REAL, classes=[0,1,2])

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
fig.suptitle(
    'Curvas ROC Multiclase (One-vs-Rest) — Evaluación in-sample\n'
    'Línea punteada = clasificador aleatorio (AUC=0.50)',
    fontsize=12, fontweight='bold'
)

COLORES_ROC = ['#e74c3c', '#3498db', '#27ae60']

def dibujar_roc_multiclase(ax, y_score_cont, nombre):
    """
    Para modelos de regresión: usa la predicción continua como
    score implícito. Para clase k, score_k = -|pred - umbral_k|
    (mayor score = más cercano al umbral de esa clase).
    Este enfoque aproxima las probabilidades sin un modelo
    clasificador explícito.
    """
    cortes_clase = [corte_bajo, corte_alto]
    umbrales_mid = [
        y.min() + (corte_bajo - y.min())/2,    # centro clase Bajo
        (corte_bajo + corte_alto)/2,            # centro clase Medio
        corte_alto + (y.max() - corte_alto)/2   # centro clase Alto
    ]
    auc_clases = []
    for k, (nombre_c, color) in enumerate(zip(NOMBRES_CLASES, COLORES_ROC)):
        score_k = -np.abs(y_score_cont - umbrales_mid[k])
        y_bin_k = (Y_CLASE_REAL == k).astype(int)
        if y_bin_k.sum() == 0 or y_bin_k.sum() == len(y_bin_k):
            continue
        fpr, tpr, _ = roc_curve(y_bin_k, score_k)
        roc_auc = auc(fpr, tpr)
        auc_clases.append(roc_auc)
        ax.plot(fpr, tpr, color=color, lw=1.8,
                label=f'{nombre_c} (AUC={roc_auc:.3f})')
    ax.plot([0,1],[0,1],'k--',lw=0.8, alpha=0.5)
    ax.set_xlim([0,1]); ax.set_ylim([0,1.02])
    ax.set_xlabel('Tasa Falsos Positivos', fontsize=8)
    ax.set_ylabel('Tasa Verdaderos Positivos', fontsize=8)
    mean_auc = np.mean(auc_clases) if auc_clases else 0
    ax.set_title(f'{nombre}\nAUC promedio: {mean_auc:.3f}',
                 fontsize=9, fontweight='bold')
    ax.legend(fontsize=8, loc='lower right')
    ax.grid(alpha=0.25)
    return mean_auc

modelos_roc = [
    ('Ridge (baseline)', y_pred_ridge),
    ('Random Forest',    y_pred_rf),
    ('XGBoost',          y_pred_xgb),
    ('LightGBM',         y_pred_lgb),
]

aucs_promedio = []
for ax, (nombre, y_score) in zip(axes, modelos_roc):
    auc_prom = dibujar_roc_multiclase(ax, y_score, nombre)
    aucs_promedio.append(auc_prom)

plt.tight_layout()
plt.savefig(os.path.join(RUTA_MASTER, 'M1_Curvas_ROC.png'),
            dpi=130, bbox_inches='tight')
plt.show()
print('✅ Curvas ROC guardadas: M1_Curvas_ROC.png')
print()
print('Resumen AUC promedio in-sample:')
for (nombre, _), auc_p in zip(modelos_roc, aucs_promedio):
    alerta = ' ← sobreajuste probable' if auc_p > 0.92 else ''
    print(f'  {nombre:<22}: AUC_prom = {auc_p:.4f}{alerta}')


In [ ]:
# ============================================================
# ANÁLISIS SHAP — Interpretabilidad del mejor modelo
#
# SHAP (SHapley Additive exPlanations):
# ──────────────────────────────────────
# SHAP asigna a cada feature una contribución aditiva a la
# predicción individual, basada en la teoría de juegos
# cooperativos (Lundberg & Lee, 2017).
# La suma de todos los SHAP values + valor base = predicción.
#
# Interpretación del gráfico beeswarm:
#   Eje X = magnitud del impacto SHAP en la predicción
#   Color = valor real de la feature (rojo=alto, azul=bajo)
#   Punto = observación individual
#   Las features más importantes aparecen arriba (mayor |SHAP|)
#
# Importancia SHAP vs importancia MDI (Mean Decrease Impurity):
#   MDI (nativa de RF) puede sobreestimar features continuas
#   con muchos valores únicos. SHAP es más confiable porque
#   evalúa el efecto marginal real en la predicción.
# ============================================================

# Seleccionar modelo con mejor RMSE_LOPO para SHAP
mejor_idx  = RESUMEN_LOPO['RMSE_mean'].idxmin()
mejor_nombre = RESUMEN_LOPO.loc[mejor_idx, 'Modelo']
print(f'Calculando SHAP para el mejor modelo LOPO: {mejor_nombre}')
print()

# Usar RF o XGB para SHAP (LGB también soportado)
if mejor_nombre == 'Random Forest':
    modelo_shap = rf_final
    explainer = shap.TreeExplainer(modelo_shap)
elif mejor_nombre == 'XGBoost':
    modelo_shap = xgb_final
    explainer = shap.TreeExplainer(modelo_shap)
elif mejor_nombre == 'LightGBM':
    modelo_shap = lgb_final
    explainer = shap.TreeExplainer(modelo_shap)
else:
    # Ridge: usar LinearExplainer
    modelo_shap = ridge_final
    explainer = shap.LinearExplainer(modelo_shap, X_scaled_final)

# Calcular SHAP values (muestra para velocidad si n > 300)
n_shap = min(len(X_arr), 300)
idx_shap = np.random.RandomState(SEMILLA).choice(len(X_arr), n_shap, replace=False)
X_shap  = X_arr[idx_shap] if mejor_nombre != 'Ridge' else X_scaled_final[idx_shap]
X_shap_df = pd.DataFrame(X_shap, columns=X.columns)

shap_values = explainer(X_shap_df)

# ── Beeswarm plot — top 20 features ──────────────────────────
fig_shap, ax_shap = plt.subplots(figsize=(10, 9))
shap.plots.beeswarm(shap_values, max_display=20, show=False)
plt.title(
    f'SHAP Beeswarm — Top 20 features más influyentes\n'
    f'Modelo: {mejor_nombre} | n={n_shap} obs.',
    fontsize=11, fontweight='bold'
)
plt.tight_layout()
plt.savefig(os.path.join(RUTA_MASTER, 'M1_SHAP_Beeswarm.png'),
            dpi=130, bbox_inches='tight')
plt.show()
print(f'✅ SHAP Beeswarm guardado: M1_SHAP_Beeswarm.png')

# Top 10 features por importancia SHAP media absoluta
importancia_shap = pd.DataFrame({
    'feature'   : X.columns,
    'shap_mean_abs': np.abs(shap_values.values).mean(axis=0)
}).sort_values('shap_mean_abs', ascending=False).head(20)

print()
print('Top 10 features por importancia SHAP:')
print(f'{"Rank":<5} {"Feature":<45} {"SHAP mean |.|":>14}')
print('-'*65)
for i, row in importancia_shap.head(10).iterrows():
    print(f'  {importancia_shap.index.get_loc(i)+1:<3}  {row.feature:<45}  {row.shap_mean_abs:>12.5f}')

# Guardar SHAP
with open(SALIDA_SHAP_PKL, 'wb') as f:
    pickle.dump({'shap_values': shap_values, 'X_shap': X_shap_df,
                 'modelo': mejor_nombre}, f)
print(f'\n✅ SHAP values guardados: {SALIDA_SHAP_PKL}')


In [ ]:
# ============================================================
# DASHBOARD COMPARATIVO — 9 paneles
# ============================================================

fig = plt.figure(figsize=(20, 18))
fig.suptitle(
    'MOD-09 — Dashboard Comparativo de Modelos\n'
    'Predicción de Rendimiento Arrocero (t/ha) | Validación LOPO',
    fontsize=13, fontweight='bold', y=0.99
)
gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)

COLORES_MOD = {
    'Ridge'       : '#95a5a6',
    'Random Forest': '#27ae60',
    'XGBoost'     : '#e67e22',
    'LightGBM'    : '#9b59b6',
}
NAMES_MOD = ['Ridge','Random Forest','XGBoost','LightGBM']
LOPO_DFS  = [df_lopo_ridge, df_lopo_rf, df_lopo_xgb, df_lopo_lgb]
PREDS_FULL = [y_pred_ridge, y_pred_rf, y_pred_xgb, y_pred_lgb]

# ── Panel 1: RMSE por fold (LOPO) ────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
for nombre, df_lopo in zip(NAMES_MOD, LOPO_DFS):
    ax1.plot(df_lopo['Periodo_test'], df_lopo['RMSE'],
             marker='o', ms=4, lw=1.5, label=nombre,
             color=COLORES_MOD[nombre])
ax1.set_title('RMSE por periodo (LOPO)', fontsize=9, fontweight='bold')
ax1.set_xlabel('Periodo test', fontsize=8)
ax1.set_ylabel('RMSE (t/ha)', fontsize=8)
ax1.tick_params(axis='x', rotation=45, labelsize=7)
ax1.legend(fontsize=7)
ax1.grid(alpha=0.3)

# ── Panel 2: R² por fold (LOPO) ──────────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
for nombre, df_lopo in zip(NAMES_MOD, LOPO_DFS):
    ax2.plot(df_lopo['Periodo_test'], df_lopo['R2'],
             marker='s', ms=4, lw=1.5, label=nombre,
             color=COLORES_MOD[nombre])
ax2.axhline(0, color='red', ls='--', lw=0.8, alpha=0.6, label='R²=0')
ax2.set_title('R² por periodo (LOPO)', fontsize=9, fontweight='bold')
ax2.set_xlabel('Periodo test', fontsize=8)
ax2.set_ylabel('R²', fontsize=8)
ax2.tick_params(axis='x', rotation=45, labelsize=7)
ax2.legend(fontsize=7)
ax2.grid(alpha=0.3)

# ── Panel 3: Comparativa RMSE mean ± std ─────────────────────
ax3 = fig.add_subplot(gs[0, 2])
rmse_means = [r['RMSE_mean'] for r in [res_ridge,res_rf,res_xgb,res_lgb]]
rmse_stds  = [r['RMSE_std']  for r in [res_ridge,res_rf,res_xgb,res_lgb]]
x_pos = range(len(NAMES_MOD))
bars = ax3.bar(x_pos, rmse_means, yerr=rmse_stds,
               color=[COLORES_MOD[n] for n in NAMES_MOD],
               edgecolor='white', capsize=5, alpha=0.85)
ax3.set_xticks(x_pos)
ax3.set_xticklabels([n.replace(' ','\n') for n in NAMES_MOD], fontsize=7)
ax3.set_ylabel('RMSE medio ± std', fontsize=8)
ax3.set_title('RMSE LOPO — Comparativa', fontsize=9, fontweight='bold')
for bar, val in zip(bars, rmse_means):
    ax3.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
             f'{val:.3f}', ha='center', va='bottom', fontsize=8)
ax3.grid(axis='y', alpha=0.3)

# ── Panel 4: Scatter real vs predicho — RF ───────────────────
ax4 = fig.add_subplot(gs[1, 0])
lim = [y_arr.min()-0.2, y_arr.max()+0.2]
ax4.scatter(y_arr, y_pred_rf, alpha=0.4, s=15, c='#27ae60')
ax4.plot(lim, lim, 'k--', lw=0.8, alpha=0.6)
ax4.set_xlim(lim); ax4.set_ylim(lim)
ax4.set_xlabel('Real (t/ha)', fontsize=8)
ax4.set_ylabel('Predicho (t/ha)', fontsize=8)
ax4.set_title(f'Real vs Predicho — Random Forest\n(in-sample  R²={m_rf_full["R2"]:.3f})',
              fontsize=9, fontweight='bold')
ax4.grid(alpha=0.25)

# ── Panel 5: Scatter real vs predicho — XGB ──────────────────
ax5 = fig.add_subplot(gs[1, 1])
ax5.scatter(y_arr, y_pred_xgb, alpha=0.4, s=15, c='#e67e22')
ax5.plot(lim, lim, 'k--', lw=0.8, alpha=0.6)
ax5.set_xlim(lim); ax5.set_ylim(lim)
ax5.set_xlabel('Real (t/ha)', fontsize=8)
ax5.set_ylabel('Predicho (t/ha)', fontsize=8)
ax5.set_title(f'Real vs Predicho — XGBoost\n(in-sample  R²={m_xgb_full["R2"]:.3f})',
              fontsize=9, fontweight='bold')
ax5.grid(alpha=0.25)

# ── Panel 6: Scatter real vs predicho — LGB ──────────────────
ax6 = fig.add_subplot(gs[1, 2])
ax6.scatter(y_arr, y_pred_lgb, alpha=0.4, s=15, c='#9b59b6')
ax6.plot(lim, lim, 'k--', lw=0.8, alpha=0.6)
ax6.set_xlim(lim); ax5.set_ylim(lim)
ax6.set_xlabel('Real (t/ha)', fontsize=8)
ax6.set_ylabel('Predicho (t/ha)', fontsize=8)
ax6.set_title(f'Real vs Predicho — LightGBM\n(in-sample  R²={m_lgb_full["R2"]:.3f})',
              fontsize=9, fontweight='bold')
ax6.grid(alpha=0.25)

# ── Panel 7: Residuales RF ────────────────────────────────────
ax7 = fig.add_subplot(gs[2, 0])
res_rf_vals = y_arr - y_pred_rf
ax7.scatter(y_pred_rf, res_rf_vals, alpha=0.35, s=12, c='#27ae60')
ax7.axhline(0, color='red', lw=0.8, ls='--')
ax7.set_xlabel('Predicho (t/ha)', fontsize=8)
ax7.set_ylabel('Residual (real - pred)', fontsize=8)
ax7.set_title('Residuales — Random Forest', fontsize=9, fontweight='bold')
ax7.grid(alpha=0.25)

# ── Panel 8: Feature importance RF (top 12) ──────────────────
ax8 = fig.add_subplot(gs[2, 1])
fi_rf = pd.Series(rf_final.feature_importances_, index=X.columns)
fi_rf_top = fi_rf.nlargest(12).sort_values()
ax8.barh(range(len(fi_rf_top)), fi_rf_top.values,
         color='#27ae60', edgecolor='white', alpha=0.85)
ax8.set_yticks(range(len(fi_rf_top)))
ax8.set_yticklabels(fi_rf_top.index, fontsize=7)
ax8.set_xlabel('Importancia (MDI)', fontsize=8)
ax8.set_title('Feature Importance RF — Top 12', fontsize=9, fontweight='bold')
ax8.grid(axis='x', alpha=0.3)

# ── Panel 9: Tabla R² LOPO ───────────────────────────────────
ax9 = fig.add_subplot(gs[2, 2])
ax9.axis('off')
tabla_data = []
for nombre, resumen in zip(NAMES_MOD, [res_ridge,res_rf,res_xgb,res_lgb]):
    tabla_data.append([
        nombre,
        f'{resumen["RMSE_mean"]:.3f}',
        f'{resumen["R2_mean"]:.3f}',
        f'{resumen["MAPE_mean"]:.1f}%'
    ])
tabla = ax9.table(
    cellText=tabla_data,
    colLabels=['Modelo','RMSE','R²','MAPE'],
    cellLoc='center', loc='center',
    bbox=[0, 0.1, 1, 0.85]
)
tabla.auto_set_font_size(False)
tabla.set_fontsize(9)
for (r, c), cell in tabla.get_celld().items():
    if r == 0:
        cell.set_facecolor('#2C3E50')
        cell.set_text_props(color='white', weight='bold')
    elif r == mejor_idx + 1:
        cell.set_facecolor('#D5F5E3')
    else:
        cell.set_facecolor('#F8F9FA' if r%2 else 'white')
    cell.set_edgecolor('#CCCCCC')
ax9.set_title('Resumen LOPO\n(verde = mejor modelo)',
              fontsize=9, fontweight='bold', pad=5)

plt.savefig(SALIDA_DASHBOARD, dpi=130, bbox_inches='tight')
plt.show()
print(f'✅ Dashboard guardado: {SALIDA_DASHBOARD}')


In [ ]:
# ============================================================
# FASE 2 — RANDOM FOREST CON REGULARIZACIÓN EXPLÍCITA
#          Y BÚSQUEDA DE HIPERPARÁMETROS CON VALIDACIÓN TEMPORAL
#
# MOTIVACIÓN:
# ───────────
# El RF del Fase 1 usó max_depth=None (árboles sin podar),
# lo que permite memorización parcial. Para reducir la
# varianza del modelo sin introducir el sobreajuste severo
# de los boosting, se aplica regularización explícita
# ajustando los hiperparámetros que controlan la complejidad
# del árbol. Además, la búsqueda de hiperparámetros se
# realiza con validación LOPO (no K-Fold aleatorio) para
# evitar leakage temporal.
#
# HIPERPARÁMETROS A BUSCAR:
# ─────────────────────────
# max_depth         : limita la profundidad del árbol
#                     (control principal de varianza)
# min_samples_leaf  : mínimo de obs. por hoja
#                     (evita hojas con 1-2 puntos = memorización)
# max_features      : fracción de features por split
#                     (aumentar reduce correlación entre árboles)
# n_estimators      : más árboles = menor varianza del ensemble
#
# ESTRATEGIA DE BÚSQUEDA:
# ────────────────────────
# Grid search manual con LOPO como evaluador.
# Se usa RMSE_LOPO_medio como métrica de selección.
# Se reporta también el RMSE_std para penalizar soluciones
# inestables (buen promedio pero alta varianza entre folds).
# ============================================================

sep = '='*65
print(sep)
print('  FASE 2 — BÚSQUEDA DE HIPERPARÁMETROS RF (VALIDACIÓN LOPO)')
print(sep)
print()
print('  Espacio de búsqueda:')

GRID_RF = {
    'n_estimators'    : [300, 500],
    'max_depth'       : [6, 10, 15, None],
    'min_samples_leaf': [3, 5, 10],
    'max_features'    : ['sqrt', 0.3],
}

combinaciones = list(ParameterGrid(GRID_RF))
print(f'  Total combinaciones: {len(combinaciones)}')
print(f'  Folds LOPO         : {PERIODOS_SERIE.nunique()}')
print(f'  Evaluaciones totales: {len(combinaciones) * PERIODOS_SERIE.nunique()}')
print()

resultados_grid = []
mejor_rmse_grid = np.inf
mejor_params_grid = None
t0_grid = time.time()

for idx_comb, params in enumerate(combinaciones):
    rf_grid = RandomForestRegressor(
        **params, random_state=SEMILLA, n_jobs=-1
    )
    lopo_grid = lopo_cv(rf_grid, X_arr, y_arr, PERIODOS_SERIE,
                        nombre=f'RF_grid_{idx_comb}')
    if not lopo_grid:
        continue
    rmse_m = np.mean([r['RMSE'] for r in lopo_grid])
    rmse_s = np.std([r['RMSE']  for r in lopo_grid])
    r2_m   = np.mean([r['R2']   for r in lopo_grid])

    resultados_grid.append({
        'n_estimators'    : params['n_estimators'],
        'max_depth'       : str(params['max_depth']),
        'min_samples_leaf': params['min_samples_leaf'],
        'max_features'    : str(params['max_features']),
        'RMSE_mean'       : rmse_m,
        'RMSE_std'        : rmse_s,
        'R2_mean'         : r2_m,
    })

    if rmse_m < mejor_rmse_grid:
        mejor_rmse_grid  = rmse_m
        mejor_params_grid = params.copy()

    # Progreso cada 5 combinaciones
    if (idx_comb + 1) % 5 == 0 or idx_comb == len(combinaciones)-1:
        elapsed = time.time() - t0_grid
        print(f'  [{idx_comb+1:2d}/{len(combinaciones)}] '
              f'RMSE actual mejor: {mejor_rmse_grid:.4f}  '
              f'({elapsed:.0f}s)')

df_grid = pd.DataFrame(resultados_grid).sort_values('RMSE_mean')

print()
print(sep)
print('  RESULTADOS BÚSQUEDA DE HIPERPARÁMETROS')
print(sep)
print()
print(f'  Top 10 combinaciones por RMSE_LOPO:')
print(f'  {"-"*75}')
print(f'  {"n_est":>5}  {"depth":>6}  {"leaf":>4}  {"feat":>5}  '
      f'{"RMSE_mean":>10}  {"RMSE_std":>9}  {"R2_mean":>8}')
for _, row in df_grid.head(10).iterrows():
    print(f'  {int(row.n_estimators):>5}  {row.max_depth:>6}  '
          f'{int(row.min_samples_leaf):>4}  {row.max_features:>5}  '
          f'{row.RMSE_mean:>10.4f}  {row.RMSE_std:>9.4f}  '
          f'{row.R2_mean:>8.4f}')

print()
print(f'  Mejores hiperparámetros encontrados:')
for k, v in mejor_params_grid.items():
    print(f'    {k:<20}: {v}')
print()
print(f'  RMSE_LOPO_mean: {mejor_rmse_grid:.4f}')
print(f'  RMSE_LOPO_std : {df_grid.iloc[0]["RMSE_std"]:.4f}')
print(f'  R²_LOPO_mean  : {df_grid.iloc[0]["R2_mean"]:.4f}')
print()
print(f'  Comparación vs RF inicial:')
print(f'    RF v1 (RMSE_LOPO): {res_rf["RMSE_mean"]:.4f}  std={res_rf["RMSE_std"]:.4f}')
print(f'    RF v2 optimizado : {mejor_rmse_grid:.4f}  std={df_grid.iloc[0]["RMSE_std"]:.4f}')
mejora = (res_rf['RMSE_mean'] - mejor_rmse_grid) / res_rf['RMSE_mean'] * 100
print(f'    Mejora RMSE      : {mejora:+.2f}%')
if mejora > 0:
    print('    ✅ Hiperparámetros óptimos mejoran la generalización')
else:
    print('    ⚠️  El RF inicial ya era cercano al óptimo para este dataset')
    print('       Esto sugiere que la señal climática tiene un límite natural')


In [ ]:
# ============================================================
# RF OPTIMIZADO — Entrenamiento final con mejores hiperparámetros
# ============================================================

print('Entrenando RF optimizado con mejores hiperparámetros...')
print()

rf_opt = RandomForestRegressor(
    **mejor_params_grid, random_state=SEMILLA, n_jobs=-1
)
rf_opt.fit(X_arr, y_arr)
y_pred_rf_opt = rf_opt.predict(X_arr)
m_rf_opt_full = calcular_metricas_regresion(y_arr, y_pred_rf_opt, 'RF_opt (full)')

# Métricas LOPO del optimizado (ya calculadas en grid)
res_rf_opt = {
    'Modelo'   : 'RF Optimizado',
    'RMSE_mean': mejor_rmse_grid,
    'RMSE_std' : df_grid.iloc[0]['RMSE_std'],
    'R2_mean'  : df_grid.iloc[0]['R2_mean'],
    'MAPE_mean': np.nan,  # se recalcula abajo
}
lopo_rf_opt = lopo_cv(rf_opt, X_arr, y_arr, PERIODOS_SERIE,
                      nombre='RF_opt_final')
res_rf_opt_full, df_lopo_rf_opt = resumir_lopo(lopo_rf_opt, 'RF Optimizado')

# Diagnóstico de regularización
print()
print('Diagnóstico de regularización:')
print(f'  RF v1 (sin regular.): RMSE_full={m_rf_full["R2"]:.4f}  '
      f'ratio={res_rf["RMSE_mean"]/m_rf_full["RMSE"]:.1f}x')
print(f'  RF v2 (regularizado): RMSE_full={m_rf_opt_full["R2"]:.4f}  '
      f'ratio={res_rf_opt_full["RMSE_mean"]/m_rf_opt_full["RMSE"]:.1f}x')
print()

ratio_v1 = res_rf['RMSE_mean'] / m_rf_full['RMSE']
ratio_v2 = res_rf_opt_full['RMSE_mean'] / m_rf_opt_full['RMSE']

if ratio_v2 < ratio_v1:
    print('  ✅ La regularización redujo la brecha in-sample vs LOPO.')
    print('     El modelo v2 memoriza menos y generaliza de forma más')
    print('     consistente. La complejidad del árbol estaba sobre-especificada.')
else:
    print('  ⚠️  La regularización no redujo significativamente el sobreajuste.')
    print('     Interpretación: el dataset es pequeño (~170 obs) y la señal')
    print('     climática es débil (correlaciones máx r≈0.44 en MOD-08).')
    print('     La varianza residual no es ruido controlable por el modelo.')

# Guardar RF optimizado
with open(SALIDA_RF_OPT_PKL, 'wb') as f:
    pickle.dump({
        'modelo'       : rf_opt,
        'hiperparametros': mejor_params_grid,
        'metricas_lopo': res_rf_opt_full,
        'features'     : X.columns.tolist(),
    }, f)
print()
print(f'✅ RF optimizado guardado: {SALIDA_RF_OPT_PKL}')


In [ ]:
# ============================================================
# SELECCION DE COMPONENTES PCA — v3.0
# Reemplaza la seleccion Top-30 features por SHAP
#
# JUSTIFICACION DEL CAMBIO A PCA:
# ---------------------------------
# El espacio de features climaticas del proyecto tiene alta
# multicolinealidad: temperatura_m1..m6, precipitacion_m1..m6,
# etc. son mediciones del mismo fenomeno en meses consecutivos
# (correlacion tipica r > 0.70 entre meses adyacentes).
#
# Problemas de la seleccion Top-30 por SHAP:
#   1. SHAP mide importancia feature por feature, ignorando la
#      informacion conjunta entre features correlacionadas.
#   2. Con n~500 obs y 70+ features correlacionadas, el RF
#      puede repartir importancia arbitrariamente entre m1-m6
#      segun el ruido de la muestra (inestable entre semillas).
#   3. Eliminar feature por feature no reduce la dimension del
#      espacio de informacion real, solo el numero de columnas.
#
# Ventajas del PCA en este contexto:
#   1. DESCORRELACIONA: los componentes son ortogonales entre si.
#      El RF recibe features independientes, lo que estabiliza
#      la importancia y reduce la varianza del modelo.
#   2. COMPRESION REAL: un componente puede capturar la varianza
#      conjunta de temperatura_m1..m6 mejor que cualquier mes
#      individual. Se reduce dimension de informacion, no solo
#      de columnas.
#   3. SELECCION OBJETIVA: el criterio de varianza acumulada
#      (90%) no depende del modelo ni de la semilla aleatoria.
#   4. INTERPRETABLE: las cargas de cada componente revelan
#      que combinacion de variables climaticas captura cada
#      patron de varianza.
#
# Criterio de seleccion de componentes:
#   Se seleccionan los primeros K componentes que cumplen:
#     a) Varianza acumulada >= PCA_VARIANZA_OBJETIVO (90%)
#     b) Cada componente aporta >= PCA_VARIANZA_MIN_COMP (1%)
#     c) K <= PCA_MAX_COMPONENTES (30)
# ============================================================

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

sep = '='*65
print(sep)
print('  SELECCION DE COMPONENTES PCA v3.0')
print(f'  Varianza objetivo : >= {PCA_VARIANZA_OBJETIVO*100:.0f}%')
print(f'  Varianza min/comp : >= {PCA_VARIANZA_MIN_COMP*100:.0f}%')
print(f'  Max componentes   : {PCA_MAX_COMPONENTES}')
print(sep)
print()

# ── Paso 1: Estandarizar features (PCA requiere escala comun) ─
# IMPORTANTE: PCA debe aplicarse sobre features estandarizadas
# para que la varianza de features de mayor magnitud
# (ej. precipitacion en mm) no domine a las de menor magnitud
# (ej. temperatura en C).
print('Paso 1: Estandarizando features para PCA...')
scaler_pca = StandardScaler()
X_scaled_pca = scaler_pca.fit_transform(X_arr)
print(f'   Shape estandarizada: {X_scaled_pca.shape[0]:,} obs x {X_scaled_pca.shape[1]} features')
print(f'   Media post-escala  : {X_scaled_pca.mean():.6f} (esperado: ~0)')
print(f'   Std  post-escala   : {X_scaled_pca.std():.6f}  (esperado: ~1)')
print()

# ── Paso 2: PCA completo — analisis de varianza explicada ─────
print('Paso 2: Ajustando PCA completo (todos los componentes)...')
pca_completo = PCA(random_state=SEMILLA)
pca_completo.fit(X_scaled_pca)

varianza_exp       = pca_completo.explained_variance_ratio_
varianza_acumulada = np.cumsum(varianza_exp)

print(f'   Total componentes posibles: {len(varianza_exp)}')
print()

# ── Paso 3: Determinar numero optimo de componentes ───────────
print('Paso 3: Determinando numero optimo de componentes...')
print()

# Criterio a: varianza acumulada >= objetivo
n_comp_varianza = int(np.argmax(varianza_acumulada >= PCA_VARIANZA_OBJETIVO) + 1)

# Criterio b: cada componente aporta >= varianza minima
n_comp_min = int(np.sum(varianza_exp >= PCA_VARIANZA_MIN_COMP))

# Criterio c: maximo permitido
N_COMPONENTES_PCA = min(
    max(n_comp_varianza, 1),
    n_comp_min,
    PCA_MAX_COMPONENTES
)

print(f'  Criterio a) Varianza acumulada >= {PCA_VARIANZA_OBJETIVO*100:.0f}%  : {n_comp_varianza} componentes')
print(f'  Criterio b) Cada comp. >= {PCA_VARIANZA_MIN_COMP*100:.0f}% varianza    : {n_comp_min} componentes')
print(f'  Criterio c) Maximo permitido              : {PCA_MAX_COMPONENTES} componentes')
print()
print(f'  => N componentes seleccionados: {N_COMPONENTES_PCA}')
print(f'  => Varianza acumulada cubierta: {varianza_acumulada[N_COMPONENTES_PCA-1]*100:.2f}%')
print()

# ── Tabla detallada de componentes seleccionados ─────────────
print('Detalle de componentes seleccionados:')
print(f'  {"Comp":>5}  {"Var. propia":>12}  {"Var. acum.":>11}  {"Feature con mayor carga"}')
print(f'  {"-"*65}')
for i in range(N_COMPONENTES_PCA):
    var_i    = varianza_exp[i] * 100
    var_acum = varianza_acumulada[i] * 100
    # Feature con mayor carga absoluta en este componente
    cargas_i = np.abs(pca_completo.components_[i])
    top_feat  = X.columns[np.argmax(cargas_i)]
    top_carga = cargas_i.max()
    print(f'  PC{i+1:>3}  {var_i:>10.2f}%  {var_acum:>10.2f}%  {top_feat}  (|carga|={top_carga:.3f})')

if N_COMPONENTES_PCA < len(varianza_exp):
    var_restante = (1 - varianza_acumulada[N_COMPONENTES_PCA-1]) * 100
    print(f'  ... ({len(varianza_exp)-N_COMPONENTES_PCA} componentes restantes: {var_restante:.2f}% varianza descartada)')

# ── Paso 4: PCA final con N componentes seleccionados ─────────
print()
print('Paso 4: Ajustando PCA final...')
pca_final = PCA(n_components=N_COMPONENTES_PCA, random_state=SEMILLA)
X_pca = pca_final.fit_transform(X_scaled_pca)

# Nombrar columnas PCA
NOMBRES_PCA = [f'PC{i+1}' for i in range(N_COMPONENTES_PCA)]
X_pca_df    = pd.DataFrame(X_pca, columns=NOMBRES_PCA)

print(f'   Shape X_pca: {X_pca_df.shape[0]:,} obs x {X_pca_df.shape[1]} componentes')
print(f'   Ratio n/p  : {X_pca_df.shape[0]/X_pca_df.shape[1]:.1f} obs/componente')
print(f'   Varianza total explicada: {pca_final.explained_variance_ratio_.sum()*100:.2f}%')
print()

# ── Paso 5: Correlacion componentes vs target ─────────────────
# Identifica cuales componentes tienen relacion con el rendimiento
print('Paso 5: Correlacion de cada componente con el target (rendimiento)...')
print()
print(f'  {"Componente":>11}  {"Var. propia":>11}  {"Corr. target":>13}  {"Relevancia"}')
print(f'  {"-"*55}')

corrs_target = []
for i, col_pca in enumerate(NOMBRES_PCA):
    corr_val = np.corrcoef(X_pca_df[col_pca], y_arr)[0, 1]
    corrs_target.append(abs(corr_val))
    var_i = pca_final.explained_variance_ratio_[i] * 100
    relevancia = 'Alta' if abs(corr_val) >= 0.20 else ('Media' if abs(corr_val) >= 0.10 else 'Baja')
    marca = ' <--' if abs(corr_val) >= 0.15 else ''
    print(f'  {col_pca:>11}  {var_i:>10.2f}%  {corr_val:>+12.4f}  {relevancia}{marca}')

print()
n_alta_corr = sum(1 for c in corrs_target if c >= 0.15)
print(f'  Componentes con |corr| >= 0.15 (alta relevancia predictiva): {n_alta_corr}')

# ── Guardar PCA para trazabilidad ────────────────────────────
df_cargas = pd.DataFrame(
    pca_final.components_.T,
    index=X.columns,
    columns=NOMBRES_PCA
)

df_varianza_pca = pd.DataFrame({
    'componente'     : NOMBRES_PCA,
    'varianza_propia': pca_final.explained_variance_ratio_ * 100,
    'varianza_acumulada': np.cumsum(pca_final.explained_variance_ratio_) * 100,
    'corr_target_abs': corrs_target,
})

with open(SALIDA_PCA_PKL, 'wb') as f:
    pickle.dump({
        'pca'              : pca_final,
        'scaler'           : scaler_pca,
        'n_componentes'    : N_COMPONENTES_PCA,
        'varianza_explicada': pca_final.explained_variance_ratio_,
        'cargas'           : df_cargas,
        'varianza_df'      : df_varianza_pca,
        'features_originales': X.columns.tolist(),
        'nombres_pca'      : NOMBRES_PCA,
    }, f)
print(f'OK PCA guardado: {SALIDA_PCA_PKL}')


In [ ]:
# ============================================================
# DASHBOARD PCA — Visualizaciones del analisis de componentes
# 4 paneles: varianza, codo, correlacion vs target, cargas
# ============================================================

fig_pca, axes_pca = plt.subplots(2, 2, figsize=(16, 12))
fig_pca.suptitle(
    f'Analisis de Componentes Principales (PCA)\n'
    f'{N_COMPONENTES_PCA} componentes seleccionados — '
    f'{pca_final.explained_variance_ratio_.sum()*100:.1f}% varianza explicada',
    fontsize=13, fontweight='bold'
)

# Panel 1: Varianza explicada por componente (barras)
ax1 = axes_pca[0, 0]
n_mostrar = min(N_COMPONENTES_PCA + 5, len(varianza_exp))
colores_comp = ['#1565C0' if i < N_COMPONENTES_PCA else '#CCCCCC'
                for i in range(n_mostrar)]
ax1.bar(range(1, n_mostrar+1),
        varianza_exp[:n_mostrar]*100,
        color=colores_comp, edgecolor='white')
ax1.axvline(N_COMPONENTES_PCA + 0.5, color='red', ls='--', lw=1.5,
            label=f'Corte: {N_COMPONENTES_PCA} componentes')
ax1.set_xlabel('Componente Principal', fontsize=9)
ax1.set_ylabel('Varianza explicada (%)', fontsize=9)
ax1.set_title('Varianza explicada por componente\n(azul = seleccionados)',
              fontsize=10, fontweight='bold')
ax1.legend(fontsize=8)
ax1.grid(axis='y', alpha=0.3)

# Panel 2: Curva de varianza acumulada (codo)
ax2 = axes_pca[0, 1]
n_total_comp = min(len(varianza_exp), 40)
ax2.plot(range(1, n_total_comp+1),
         varianza_acumulada[:n_total_comp]*100,
         'o-', color='#1565C0', ms=4, lw=2)
ax2.axhline(PCA_VARIANZA_OBJETIVO*100, color='orange', ls='--', lw=1.2,
            label=f'Objetivo: {PCA_VARIANZA_OBJETIVO*100:.0f}%')
ax2.axvline(N_COMPONENTES_PCA, color='red', ls='--', lw=1.5,
            label=f'Seleccionados: {N_COMPONENTES_PCA}')
ax2.fill_between(range(1, N_COMPONENTES_PCA+1),
                 varianza_acumulada[:N_COMPONENTES_PCA]*100,
                 alpha=0.15, color='#1565C0')
ax2.set_xlabel('N componentes', fontsize=9)
ax2.set_ylabel('Varianza acumulada (%)', fontsize=9)
ax2.set_title('Curva de varianza acumulada\n(criterio de seleccion)',
              fontsize=10, fontweight='bold')
ax2.legend(fontsize=8)
ax2.grid(alpha=0.3)

# Panel 3: Correlacion de cada componente vs target
ax3 = axes_pca[1, 0]
colores_corr = ['#e74c3c' if c >= 0.15 else ('#f39c12' if c >= 0.10 else '#95a5a6')
                for c in corrs_target]
ax3.barh(range(N_COMPONENTES_PCA),
         corrs_target,
         color=colores_corr, edgecolor='white')
ax3.axvline(0.15, color='red', ls='--', lw=1, alpha=0.7,
            label='|corr|=0.15 (alta relevancia)')
ax3.axvline(0.10, color='orange', ls='--', lw=1, alpha=0.7,
            label='|corr|=0.10 (media relevancia)')
ax3.set_yticks(range(N_COMPONENTES_PCA))
ax3.set_yticklabels([f'PC{i+1}' for i in range(N_COMPONENTES_PCA)], fontsize=8)
ax3.set_xlabel('|Correlacion con Rendimiento (t/ha)|', fontsize=9)
ax3.set_title('Relevancia predictiva de cada componente\n(correlacion vs target)',
              fontsize=10, fontweight='bold')
ax3.legend(fontsize=7)
ax3.grid(axis='x', alpha=0.3)

# Panel 4: Heatmap de cargas (top componentes vs top features originales)
ax4 = axes_pca[1, 1]
n_comp_heat  = min(N_COMPONENTES_PCA, 10)
n_feat_heat  = min(X.shape[1], 20)
# Seleccionar las features con mayor varianza de carga entre componentes
varianza_cargas = df_cargas.iloc[:, :n_comp_heat].var(axis=1)
top_feats_heat  = varianza_cargas.nlargest(n_feat_heat).index.tolist()
heat_data = df_cargas.loc[top_feats_heat, NOMBRES_PCA[:n_comp_heat]].values
im = ax4.imshow(heat_data, cmap='RdBu_r', aspect='auto', vmin=-1, vmax=1)
ax4.set_xticks(range(n_comp_heat))
ax4.set_xticklabels([f'PC{i+1}' for i in range(n_comp_heat)], fontsize=8)
ax4.set_yticks(range(len(top_feats_heat)))
ax4.set_yticklabels([f[:30] for f in top_feats_heat], fontsize=6)
ax4.set_title(f'Cargas PCA — Top {n_feat_heat} features\n'
              f'(rojo=carga positiva, azul=negativa)',
              fontsize=10, fontweight='bold')
plt.colorbar(im, ax=ax4, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.savefig(os.path.join(RUTA_MASTER, 'M1_PCA_Dashboard.png'),
            dpi=130, bbox_inches='tight')
plt.show()
print('OK Dashboard PCA guardado: M1_PCA_Dashboard.png')

# ── Tabla de features mas influyentes por componente ─────────
print()
print('Top 3 features con mayor carga en cada componente PCA seleccionado:')
print(f'  {"Componente":>11}  {"Var.":>6}  {"Top-1 feature":<35}  '
      f'{"Top-2 feature":<35}  Top-3 feature')
print(f'  {"-"*120}')
for i in range(N_COMPONENTES_PCA):
    cargas_i  = np.abs(pca_final.components_[i])
    top3_idx  = cargas_i.argsort()[::-1][:3]
    top3_feat = [(X.columns[j], cargas_i[j]) for j in top3_idx]
    var_i     = pca_final.explained_variance_ratio_[i]*100
    print(f'  PC{i+1:>3} ({var_i:4.1f}%)  '
          + '  '.join(f'{f:<30} ({c:.3f})' for f, c in top3_feat))


In [ ]:
# ============================================================
# MODELO FINAL — RF sobre componentes PCA
# Validacion LOPO + Dashboard + SHAP sobre componentes
# ============================================================

sep = '='*65
print(sep)
print(f'  MODELO FINAL — RF sobre {N_COMPONENTES_PCA} Componentes PCA')
print(sep)
print()

X_pca_arr = X_pca_df.values.astype(float)

# ── 1. Validacion LOPO sobre espacio PCA ─────────────────────
print('1. Validacion LOPO — RF sobre componentes PCA:')
print()
print('   NOTA METODOLOGICA: el scaler y el PCA se ajustan dentro')
print('   de cada fold de entrenamiento para evitar data leakage.')
print()

def lopo_cv_pca(modelo_cls, X_orig, y, periodos,
                n_componentes, varianza_objetivo, semilla=42):
    '''
    LOPO con PCA ajustado DENTRO de cada fold.

    Razon: si el PCA se ajusta sobre todo el dataset antes del
    loop LOPO, el fold de test contamina la transformacion
    (data leakage). El scaler y PCA deben ajustarse solo con
    los datos de entrenamiento de cada fold y luego aplicarse
    al fold de test.

    Flujo por fold:
      1. Separar train / test por periodo
      2. Ajustar StandardScaler en train, transformar train y test
      3. Ajustar PCA en train escalado, transformar train y test
      4. Entrenar RF en train PCA
      5. Predecir en test PCA
      6. Calcular metricas
    '''
    import copy
    periodos_unicos = sorted(periodos.unique())
    resultados = []

    for per_test in periodos_unicos:
        mask_test  = (periodos == per_test)
        mask_train = ~mask_test

        X_tr_orig = X_orig[mask_train]
        X_te_orig = X_orig[mask_test]
        y_tr = y[mask_train]
        y_te = y[mask_test]

        if len(y_te) == 0:
            continue

        # Escalar dentro del fold
        sc_fold = StandardScaler()
        X_tr_sc = sc_fold.fit_transform(X_tr_orig)
        X_te_sc = sc_fold.transform(X_te_orig)

        # PCA dentro del fold
        n_comp_fold = min(n_componentes, X_tr_sc.shape[0]-1, X_tr_sc.shape[1])
        pca_fold = PCA(n_components=n_comp_fold, random_state=semilla)
        X_tr_pca = pca_fold.fit_transform(X_tr_sc)
        X_te_pca = pca_fold.transform(X_te_sc)

        modelo = copy.deepcopy(modelo_cls)
        try:
            modelo.fit(X_tr_pca, y_tr)
            y_pred = modelo.predict(X_te_pca)
            metricas = {
                'RMSE': float(np.sqrt(mean_squared_error(y_te, y_pred))),
                'MAE' : float(mean_absolute_error(y_te, y_pred)),
                'R2'  : float(r2_score(y_te, y_pred)),
                'MAPE': float(mean_absolute_percentage_error(y_te, y_pred)*100),
                'Periodo_test': per_test,
                'n_test'      : int(len(y_te)),
                'n_train'     : int(len(y_tr)),
                'n_comp_fold' : n_comp_fold,
            }
            resultados.append(metricas)
        except Exception as e:
            print(f'     Fold {per_test}: error — {e}')

    return resultados


rf_pca_model = RandomForestRegressor(**mejor_params_grid, random_state=SEMILLA, n_jobs=-1)

lopo_pca = lopo_cv_pca(
    rf_pca_model, X_arr, y_arr, PERIODOS_SERIE,
    n_componentes=N_COMPONENTES_PCA,
    varianza_objetivo=PCA_VARIANZA_OBJETIVO,
    semilla=SEMILLA
)
res_pca, df_lopo_pca = resumir_lopo(lopo_pca, f'RF_PCA ({N_COMPONENTES_PCA} comp.)')

# ── 2. Tabla comparativa: RF original vs RF+PCA ───────────────
print()
print('2. Comparacion de modelos RF (validacion LOPO):')
print(f'  {"Modelo":<32}  {"RMSE_mean":>10}  {"RMSE_std":>9}  {"R2_mean":>8}  {"n_features":>10}')
print(f'  {"-"*75}')
for nombre_m, resumen_m, n_feat in [
    ('RF v1 (todas features)',       res_rf,          X.shape[1]),
    ('RF v2 (regularizado)',         res_rf_opt_full, X.shape[1]),
    (f'RF v3 (PCA {N_COMPONENTES_PCA} comp.)', res_pca, N_COMPONENTES_PCA),
]:
    mejor_rmse_todos = min(res_rf['RMSE_mean'],
                          res_rf_opt_full['RMSE_mean'],
                          res_pca['RMSE_mean'])
    marca = ' MEJOR' if abs(resumen_m['RMSE_mean'] - mejor_rmse_todos) < 0.0001 else ''
    print(f'  {nombre_m:<32}  {resumen_m["RMSE_mean"]:>10.4f}  '
          f'{resumen_m["RMSE_std"]:>9.4f}  '
          f'{resumen_m["R2_mean"]:>8.4f}  {n_feat:>10}{marca}')

# ── 3. Entrenamiento final RF+PCA sobre dataset completo ──────
print()
print('3. Entrenamiento final RF+PCA sobre dataset completo...')
rf_pca_final = RandomForestRegressor(**mejor_params_grid, random_state=SEMILLA, n_jobs=-1)
rf_pca_final.fit(X_pca_arr, y_arr)
y_pred_pca = rf_pca_final.predict(X_pca_arr)
m_pca_full  = calcular_metricas_regresion(y_arr, y_pred_pca, 'RF_PCA (full)')

ratio_pca = res_pca['RMSE_mean'] / m_pca_full['RMSE'] if m_pca_full['RMSE'] > 0 else 999
ratio_v1  = res_rf['RMSE_mean']  / m_rf_full['RMSE']  if m_rf_full['RMSE']  > 0 else 999
print()
print('Diagnostico de sobreajuste:')
print(f'  RF v1 (todas features): ratio RMSE_LOPO/full = {ratio_v1:.1f}x')
print(f'  RF v3 (PCA):            ratio RMSE_LOPO/full = {ratio_pca:.1f}x')
if ratio_pca < ratio_v1:
    print('  OK PCA redujo el sobreajuste respecto al RF v1.')
    print('     Los componentes ortogonales del PCA reducen la varianza')
    print('     del modelo al eliminar la multicolinealidad entre features.')
else:
    print('  AVISO: PCA no redujo el sobreajuste respecto al RF v1.')
    print('     Posible causa: la varianza explicada por PCA no es la')
    print('     varianza relevante para predecir rendimiento.')

Y_PRED_PCA_C = pred_a_clases(y_pred_pca, corte_bajo, corte_alto)

# ── 4. Dashboard LOPO: RF v1 vs RF+PCA ───────────────────────
fig_lopo_pca, axes_lopo = plt.subplots(1, 2, figsize=(14, 5))
fig_lopo_pca.suptitle(
    f'Comparacion LOPO — RF v1 (todas features) vs RF v3 (PCA {N_COMPONENTES_PCA} componentes)',
    fontsize=11, fontweight='bold'
)
for ax, metrica, ylabel in zip(axes_lopo, ['RMSE','R2'], ['RMSE (t/ha)','R2']):
    ax.plot(df_lopo_rf['Periodo_test'],  df_lopo_rf[metrica],
            'o-', ms=5, lw=1.8, color='#27ae60', label='RF v1 (todas features)')
    ax.plot(df_lopo_pca['Periodo_test'], df_lopo_pca[metrica],
            's--', ms=5, lw=1.8, color='#e74c3c',
            label=f'RF v3 (PCA {N_COMPONENTES_PCA} comp.)')
    if metrica == 'R2':
        ax.axhline(0, color='gray', ls=':', lw=0.8, alpha=0.6)
    ax.set_xlabel('Periodo test', fontsize=9)
    ax.set_ylabel(ylabel, fontsize=9)
    ax.set_title(f'{metrica} por fold LOPO', fontsize=9, fontweight='bold')
    ax.tick_params(axis='x', rotation=45, labelsize=7)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(RUTA_MASTER, 'M1_RF_PCA_LOPO_Comparacion.png'),
            dpi=130, bbox_inches='tight')
plt.show()
print('OK Comparacion LOPO guardada: M1_RF_PCA_LOPO_Comparacion.png')

# ── 5. Matriz confusion RF+PCA ────────────────────────────────
fig_cm_pca, axes_cm = plt.subplots(1, 3, figsize=(15, 5))
fig_cm_pca.suptitle(
    'Matrices de Confusion — Evolucion de la regularizacion RF\n'
    'v1 (todas features) | v2 (regularizado) | v3 (PCA)',
    fontsize=11, fontweight='bold'
)
cmap_m = LinearSegmentedColormap.from_list('custom', ['#FFFFFF','#1565C0'], N=256)
for ax, (nombre_m, y_pred_c) in zip(axes_cm, [
    ('RF v1 (todas features)', Y_PRED_RF_C),
    ('RF v2 (regularizado)',   pred_a_clases(y_pred_rf_opt, corte_bajo, corte_alto)),
    (f'RF v3 (PCA {N_COMPONENTES_PCA} comp.)', Y_PRED_PCA_C),
]):
    cm  = confusion_matrix(Y_CLASE_REAL, y_pred_c, labels=[0,1,2])
    acc = np.trace(cm)/cm.sum()
    adj = (cm[0,1]+cm[1,0]+cm[1,2]+cm[2,1])/cm.sum()
    im  = ax.imshow(cm, cmap=cmap_m, aspect='auto')
    ax.set_xticks([0,1,2]); ax.set_yticks([0,1,2])
    ax.set_xticklabels(NOMBRES_CLASES, fontsize=8)
    ax.set_yticklabels(NOMBRES_CLASES, fontsize=8)
    ax.set_xlabel('Predicho', fontsize=8)
    ax.set_ylabel('Real', fontsize=8)
    ax.set_title(f'{nombre_m}\nAcc={acc:.1%}  Err_adj={adj:.1%}',
                 fontsize=8, fontweight='bold')
    mx = cm.max()
    for ii in range(3):
        for jj in range(3):
            col = 'white' if cm[ii,jj] > mx*0.6 else 'black'
            ax.text(jj, ii, str(cm[ii,jj]), ha='center', va='center',
                    color=col, fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(RUTA_MASTER, 'M1_RF_PCA_Matrices.png'),
            dpi=130, bbox_inches='tight')
plt.show()
print('OK Matrices de confusion guardadas: M1_RF_PCA_Matrices.png')

# Guardamos referencia para uso en celda 14
TOP_FEATURES_NOMBRES = NOMBRES_PCA   # compatibilidad con celda 14
TOP_N_FEATURES       = N_COMPONENTES_PCA
X_top     = X_pca_df
X_top_arr = X_pca_arr
rf_top30  = rf_pca_final   # alias para compatibilidad con celdas 14-17
res_top30 = res_pca
df_lopo_top30 = df_lopo_pca
y_pred_top30  = y_pred_pca
m_top30_full  = m_pca_full


In [ ]:
# ============================================================
# SHAP FINAL + VISUALIZACIONES DEL MODELO RF Top-30
# ============================================================

print(f'Calculando SHAP del modelo RF+PCA ({N_COMPONENTES_PCA} componentes)...')
print()
print('NOTA: Los SHAP values se calculan sobre los componentes PCA,')
print('no sobre las features originales. El beeswarm muestra que')
print('componentes (combinaciones de variables climaticas) son')
print('mas influyentes en la prediccion de rendimiento.')

print()

# SHAP del RF top-30
explainer_top30 = shap.TreeExplainer(rf_pca_final)
n_shap_t = min(len(X_top_arr), 300)
idx_shap_t = np.random.RandomState(SEMILLA).choice(len(X_top_arr), n_shap_t, replace=False)
X_shap_t_df = pd.DataFrame(X_top_arr[idx_shap_t], columns=TOP_FEATURES_NOMBRES)
shap_values_top30 = explainer_top30(X_shap_t_df)

# ── Beeswarm SHAP top-30 ─────────────────────────────────────
fig_sw, ax_sw = plt.subplots(figsize=(10, 10))
shap.plots.beeswarm(shap_values_top30, max_display=TOP_N_FEATURES, show=False)
plt.title(
    f'SHAP Beeswarm — RF+PCA Modelo Final ({N_COMPONENTES_PCA} componentes)\n'
    f'Sin identificadores geográficos | n={n_shap_t} obs.',
    fontsize=11, fontweight='bold'
)
plt.tight_layout()
plt.savefig(os.path.join(RUTA_MASTER, 'M1_RF_PCA_SHAP.png'),
            dpi=130, bbox_inches='tight')
plt.show()
print('✅ SHAP Top-30 guardado')

# ── Dashboard final: scatter + residuales + importancia ──────
fig_fin, axes_fin = plt.subplots(1, 3, figsize=(18, 6))
fig_fin.suptitle(
    f'Modelo Final — RF+PCA ({N_COMPONENTES_PCA} componentes)\n'
    f'RMSE_LOPO={res_top30["RMSE_mean"]:.4f} t/ha  |  '
    f'R²_LOPO={res_top30["R2_mean"]:.4f}',
    fontsize=11, fontweight='bold'
)

# Panel 1: Real vs Predicho
lim_fin = [y_arr.min()-0.3, y_arr.max()+0.3]
axes_fin[0].scatter(y_arr, y_pred_top30, alpha=0.5, s=20, c='#1565C0', zorder=3)
axes_fin[0].plot(lim_fin, lim_fin, 'k--', lw=0.9, alpha=0.5)
axes_fin[0].set_xlim(lim_fin); axes_fin[0].set_ylim(lim_fin)
axes_fin[0].set_xlabel('Rendimiento real (t/ha)', fontsize=9)
axes_fin[0].set_ylabel('Rendimiento predicho (t/ha)', fontsize=9)
axes_fin[0].set_title(
    f'Real vs Predicho (in-sample)\nR²={m_top30_full["R2"]:.3f}  RMSE={m_top30_full["RMSE"]:.3f}',
    fontsize=9, fontweight='bold')
axes_fin[0].grid(alpha=0.25)

# Panel 2: Residuales
res_top30_vals = y_arr - y_pred_top30
axes_fin[1].scatter(y_pred_top30, res_top30_vals, alpha=0.45, s=15, c='#1565C0')
axes_fin[1].axhline(0, color='red', lw=0.9, ls='--')
# Líneas ±1 RMSE_LOPO para referencia
rmse_ref = res_top30['RMSE_mean']
axes_fin[1].axhline(+rmse_ref,  color='orange', lw=0.8, ls=':', alpha=0.7,
                     label=f'+RMSE_LOPO ({rmse_ref:.2f})')
axes_fin[1].axhline(-rmse_ref,  color='orange', lw=0.8, ls=':', alpha=0.7,
                     label=f'-RMSE_LOPO ({rmse_ref:.2f})')
axes_fin[1].set_xlabel('Predicho (t/ha)', fontsize=9)
axes_fin[1].set_ylabel('Residual (real - pred)', fontsize=9)
axes_fin[1].set_title('Residuales\nLíneas naranjas = ±RMSE_LOPO', fontsize=9, fontweight='bold')
axes_fin[1].legend(fontsize=7)
axes_fin[1].grid(alpha=0.25)

# Panel 3: Importancia SHAP top-15
shap_fin_imp = pd.DataFrame({
    'feature'      : TOP_FEATURES_NOMBRES,
    'shap_mean_abs': np.abs(shap_values_top30.values).mean(axis=0)
}).sort_values('shap_mean_abs').tail(15)
axes_fin[2].barh(range(len(shap_fin_imp)), shap_fin_imp['shap_mean_abs'],
                  color='#1565C0', edgecolor='white', alpha=0.85)
axes_fin[2].set_yticks(range(len(shap_fin_imp)))
axes_fin[2].set_yticklabels(shap_fin_imp['feature'], fontsize=7)
axes_fin[2].set_xlabel('SHAP mean |.| (t/ha)', fontsize=8)
axes_fin[2].set_title(f'Top 15 features por SHAP\nModelo RF Final', fontsize=9, fontweight='bold')
axes_fin[2].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(RUTA_MASTER, 'M1_RF_PCA_Dashboard.png'),
            dpi=130, bbox_inches='tight')
plt.show()
print('✅ Dashboard RF final guardado')

# Guardar modelo top-30 + SHAP
with open(SALIDA_RF_TOP30_PKL, 'wb') as f:
    pickle.dump({
        'modelo'         : rf_top30,
        'hiperparametros': mejor_params_grid,
        'features'       : TOP_FEATURES_NOMBRES,
        'n_features'     : TOP_N_FEATURES,
        'metricas_lopo'  : res_top30,
        'shap_values'    : shap_values_top30,
        'X_shap'         : X_shap_t_df,
        'corte_bajo'     : float(corte_bajo),
        'corte_alto'     : float(corte_alto),
    }, f)
with open(SALIDA_SHAP_PKL, 'wb') as f:
    pickle.dump({'shap_values': shap_values_top30, 'X_shap': X_shap_t_df,
                 'modelo': f'RF_Top{TOP_N_FEATURES}'}, f)
print(f'✅ Modelos guardados: {SALIDA_RF_TOP30_PKL}')


In [ ]:
# ============================================================
# GUARDAR RESULTADOS Y MODELO FINAL
# ============================================================

print('💾 Guardando resultados...')

# ── Excel con 5 hojas ─────────────────────────────────────────
with pd.ExcelWriter(SALIDA_RESULTADOS, engine='openpyxl') as writer:

    # Hoja 1: Comparativa LOPO resumida
    RESUMEN_LOPO.to_excel(writer, sheet_name='Comparativa_LOPO', index=False)

    # Hoja 2: Detalle LOPO por fold (todos los modelos)
    df_lopo_completo = pd.concat([
        df_lopo_ridge.assign(Modelo='Ridge'),
        df_lopo_rf.assign(Modelo='Random Forest'),
        df_lopo_xgb.assign(Modelo='XGBoost'),
        df_lopo_lgb.assign(Modelo='LightGBM'),
    ], ignore_index=True)
    df_lopo_completo.to_excel(writer, sheet_name='LOPO_por_Fold', index=False)

    # Hoja 3: Predicciones in-sample
    df_predicciones = pd.DataFrame({
        'Periodo'           : df_model['Periodo'].values,
        'Departamento'      : df_model['Departamento'].values,
        'Municipio'         : df_model['Municipio'].values,
        'Real_tpha'         : y_arr,
        'Clase_Real'        : [NOMBRES_CLASES[c] for c in Y_CLASE_REAL],
        'Pred_Ridge'        : y_pred_ridge.round(3),
        'Pred_RF'           : y_pred_rf.round(3),
        'Pred_XGB'          : y_pred_xgb.round(3),
        'Pred_LGB'          : y_pred_lgb.round(3),
        'Clase_Ridge'       : [NOMBRES_CLASES[c] for c in Y_PRED_RIDGE_C],
        'Clase_RF'          : [NOMBRES_CLASES[c] for c in Y_PRED_RF_C],
        'Clase_XGB'         : [NOMBRES_CLASES[c] for c in Y_PRED_XGB_C],
        'Clase_LGB'         : [NOMBRES_CLASES[c] for c in Y_PRED_LGB_C],
    })
    df_predicciones.to_excel(writer, sheet_name='Predicciones', index=False)

    # Hoja 4: Feature importance RF
    fi_df = pd.DataFrame({
        'feature'       : X.columns,
        'importance_rf' : rf_final.feature_importances_,
        'shap_mean_abs' : importancia_shap.set_index('feature').reindex(X.columns)['shap_mean_abs'].values
    }).sort_values('importance_rf', ascending=False)
    fi_df.to_excel(writer, sheet_name='Feature_Importance', index=False)

    # Hoja 5: RF top-30 resultados LOPO por fold
    df_lopo_top30_export = df_lopo_top30.copy()
    df_lopo_top30_export['Modelo'] = f'RF_Top{TOP_N_FEATURES}'
    df_lopo_top30_export.to_excel(writer, sheet_name='RF_Top30_LOPO', index=False)

    # Hoja 6: Componentes PCA y su varianza
    df_varianza_pca.to_excel(writer, sheet_name='PCA_Componentes', index=False)
    df_cargas.reset_index().rename(columns={'index':'feature'})\
             .to_excel(writer, sheet_name='PCA_Cargas', index=False)

    # Hoja 7: Hiperparámetros usados
    df_params = pd.DataFrame([
        {'Modelo':'Ridge',        'Parámetro':'alpha',              'Valor': PARAMS_RIDGE['alpha']},
        {'Modelo':'Random Forest','Parámetro':'n_estimators',        'Valor': PARAMS_RF['n_estimators']},
        {'Modelo':'Random Forest','Parámetro':'max_features',        'Valor': PARAMS_RF['max_features']},
        {'Modelo':'Random Forest','Parámetro':'min_samples_leaf',    'Valor': PARAMS_RF['min_samples_leaf']},
        {'Modelo':'XGBoost',      'Parámetro':'n_estimators',        'Valor': PARAMS_XGB['n_estimators']},
        {'Modelo':'XGBoost',      'Parámetro':'learning_rate',       'Valor': PARAMS_XGB['learning_rate']},
        {'Modelo':'XGBoost',      'Parámetro':'max_depth',           'Valor': PARAMS_XGB['max_depth']},
        {'Modelo':'LightGBM',     'Parámetro':'num_leaves',          'Valor': PARAMS_LGB['num_leaves']},
        {'Modelo':'LightGBM',     'Parámetro':'learning_rate',       'Valor': PARAMS_LGB['learning_rate']},
    ])
    df_params.to_excel(writer, sheet_name='Hiperparametros', index=False)

print(f'   ✅ Excel → {SALIDA_RESULTADOS}  (7 hojas)')

# ── Guardar modelo ganador ─────────────────────────────────────
# En v2: el modelo final SIEMPRE es el RF Top-30 (decisión de diseño)
# independientemente del RMSE_LOPO del grid de 4 modelos.
modelo_ganador_obj = rf_top30
mejor_nombre_final = f'RF_Top{TOP_N_FEATURES}_Regularizado'

with open(SALIDA_MODELO_PKL, 'wb') as f:
    pickle.dump({
        'modelo'      : modelo_ganador_obj,
        'nombre'         : mejor_nombre_final,
        'features'       : TOP_FEATURES_NOMBRES,
        'n_features'     : TOP_N_FEATURES,
        'hiperparametros': mejor_params_grid,
        'metricas_lopo'  : res_top30,
        'corte_bajo'  : float(corte_bajo),
        'corte_alto'  : float(corte_alto),
    }, f)
print(f'   ✅ Modelo final → {SALIDA_MODELO_PKL}')


In [ ]:
# ============================================================
# INTERPRETACIÓN CRÍTICA Y SELECCIÓN DEL MODELO FINAL
# ============================================================

sep = '='*70
print(sep)
print('  INTERPRETACIÓN CRÍTICA Y CONCLUSIONES — MOD-09')
print(sep)

print()
print('SECCIÓN 1 — VALIDACIÓN LOPO: ¿Qué tan bien generalizan los modelos?')
print('-'*70)
print()
print('La validación Leave-One-Period-Out mide la capacidad de cada modelo')
print('para predecir el rendimiento de un semestre COMPLETO que no fue parte')
print('del entrenamiento. Esta es la métrica más honesta del proyecto.')
print()
print('Resultados LOPO:')
for nombre, resumen in zip(NAMES_MOD, [res_ridge,res_rf,res_xgb,res_lgb]):
    rmse_m = resumen['RMSE_mean']
    r2_m   = resumen['R2_mean']
    mape_m = resumen['MAPE_mean']
    rmse_s = resumen['RMSE_std']
    r2_s   = resumen['R2_std']
    print(f'  {nombre:<22}: RMSE={rmse_m:.4f}±{rmse_s:.4f}  R²={r2_m:.4f}±{r2_s:.4f}  MAPE={mape_m:.1f}%')

print()
mean_y = y_arr.mean()
print(f'  Media del target: {mean_y:.4f} t/ha')
for nombre, resumen in zip(NAMES_MOD, [res_ridge,res_rf,res_xgb,res_lgb]):
    rmse_pct = resumen['RMSE_mean'] / mean_y * 100
    print(f'  {nombre:<22}: RMSE como % de la media = {rmse_pct:.1f}%')

print()
print('ALERTA DE SOBREAJUSTE:')
print('  Si RMSE_in-sample << RMSE_LOPO para un modelo, ese modelo')
print('  está memorizando el dataset, no generalizando.')
print()
for nombre, m_full, resumen in zip(
        NAMES_MOD,
        [m_ridge_full, m_rf_full, m_xgb_full, m_lgb_full],
        [res_ridge, res_rf, res_xgb, res_lgb]):
    ratio = resumen['RMSE_mean'] / m_full['RMSE'] if m_full['RMSE'] > 0 else 999
    alerta = ' ← SOBREAJUSTE CONFIRMADO' if ratio > 5 else (
             ' ← sobreajuste leve'       if ratio > 2 else ' ✓ consistente')
    print(f'  {nombre:<22}: RMSE_full={m_full["RMSE"]:.4f}  '
          f'RMSE_LOPO={resumen["RMSE_mean"]:.4f}  '
          f'ratio={ratio:.1f}{alerta}')

print()
print('SECCIÓN 2 — MATRICES DE CONFUSIÓN: ¿Cómo se distribuyen los errores?')
print('-'*70)
print()
print('Las matrices de confusión (in-sample) revelan si los errores del')
print('modelo son "tolerables" (confusión entre clases adyacentes) o')
print('"críticos" (confusión entre clases extremas Bajo↔Alto).')
print()
for nombre, y_pred_c in [('Ridge', Y_PRED_RIDGE_C),('Random Forest', Y_PRED_RF_C),
                          ('XGBoost', Y_PRED_XGB_C),('LightGBM', Y_PRED_LGB_C)]:
    cm = confusion_matrix(Y_CLASE_REAL, y_pred_c, labels=[0,1,2])
    acc  = np.trace(cm)/cm.sum()
    dist = (cm[0,2]+cm[2,0])/cm.sum()
    if acc > 0.95:
        interp = 'Accuracy perfecta in-sample confirma sobreajuste severo'
    elif dist > 0.05:
        interp = f'Errores distantes ({dist:.1%}): predice Bajo cuando es Alto o viceversa'
    elif acc > 0.70:
        interp = 'Buena discriminación, errores mayormente adyacentes (tolerables)'
    else:
        interp = 'Discriminación moderada'
    print(f'  {nombre:<22}: Acc={acc:.1%}  Err_distante={dist:.1%}  → {interp}')

print()
print('SECCIÓN 3 — CURVAS ROC: ¿Qué tan bien discrimina cada clase?')
print('-'*70)
print()
print('AUC in-sample cercano a 1.0 en modelos de árbol NO implica')
print('buen desempeño fuera de muestra. Es esperado y consecuencia del')
print('sobreajuste. El AUC debe interpretarse junto con RMSE_LOPO.')
print()
for (nombre, _), auc_p in zip(modelos_roc, aucs_promedio):
    if auc_p > 0.95:
        interp = 'Sobreajuste: discrimina perfecto en muestra, no fuera'
    elif auc_p > 0.80:
        interp = 'Buena discriminación, plausible si LOPO también es bueno'
    else:
        interp = 'Discriminación débil incluso in-sample'
    print(f'  {nombre:<22}: AUC_prom={auc_p:.4f}  → {interp}')

print()
print('SECCIÓN 4 — SELECCIÓN DEL MODELO RECOMENDADO')
print('-'*70)
print()
print(f'  Modelo seleccionado por RMSE_LOPO mínimo: {mejor_nombre}')
print()

# Análisis crítico por modelo
print('  Análisis crítico por modelo:')
print()
print('  [Ridge] Baseline lineal:')
print('    Pro  : Interpretable, estable, sin hiperparámetros complejos.')
print('    Contra: Asume linealidad entre clima y rendimiento, incorrecta')
print('            cuando existen umbrales agronómicos (estrés térmico).')
print('    Veredicto: Útil solo como referencia mínima de error.')
print()
print('  [Random Forest]:')
print('    Pro  : Maneja multicolinealidad entre meses m1-m6, estable')
print('           con n<500, feature importance interpretable, robusto')
print('           a outliers sin normalización.')
print('    Contra: Si RMSE_LOPO >> RMSE_full hay sobreajuste por')
print('            profundidad ilimitada; solución: min_samples_leaf≥3.')
print('    Veredicto: Modelo principal si RMSE_LOPO es el más bajo.')
print()
print('  [XGBoost]:')
print('    Pro  : Regularización L1/L2, early stopping, SHAP nativo.')
print('    Contra: Con n~500 y 70+ features, el boosting secuencial')
print('           tiende a memorizar; max_depth=4 y subsample=0.8')
print('           mitigan pero no eliminan el riesgo.')
print('    Veredicto: Competitivo si RMSE_LOPO es comparable a RF.')
print()
print('  [LightGBM]:')
print('    Pro  : Más rápido que XGB, leaf-wise puede capturar')
print('           relaciones no lineales complejas.')
print('    Contra: Leaf-wise con num_leaves=15 y n<500 sobreajusta')
print('           más agresivamente que level-wise (XGB/RF).')
print('           Alta sensibilidad a hiperparámetros con datos pequeños.')
print('    Veredicto: Solo preferible si el dataset crece >2000 obs.')
print()
print('CONCLUSIÓN FINAL:')
print()
print(f'  El modelo con mejor balance generalización/interpretabilidad')
print(f'  según la validación LOPO es: [{mejor_nombre}]')
print()
print('  Sin embargo, se recomienda interpretar este resultado con las')
print('  siguientes advertencias críticas:')
print()
print('  1. TAMAÑO DE MUESTRA: Con ~500 obs y 10 folds LOPO, cada fold')
print('     entrena con ~450 obs y testea con ~50. La varianza de las')
print('     métricas es alta (ver RMSE_std). Las diferencias de RMSE')
print('     entre modelos pueden NO ser estadísticamente significativas.')
print()
print('  2. SOBREAJUSTE: Todos los modelos de árbol muestran RMSE_full')
print('     << RMSE_LOPO, lo que confirma que el aprendizaje en muestra')
print('     es mucho mejor que la generalización real.')
print()
print('  3. VARIABLES CLIMÁTICAS: Las features más importantes por SHAP')
print('     deben validarse con expertos agronómicos del IDEAM/CIAT para')
print('     confirmar que la importancia estadística tiene sentido causal.')
print()
print('  4. ANÁLISIS RF v3 (Top-30 features, regularizado):')
print()
print(f'     RMSE_LOPO v1 (todas features)   : {res_rf["RMSE_mean"]:.4f} t/ha')
print(f'     RMSE_LOPO v2 (regularizado)     : {res_rf_opt_full["RMSE_mean"]:.4f} t/ha')
print(f'     RMSE_LOPO v3 (top-{TOP_N_FEATURES} features)   : {res_top30["RMSE_mean"]:.4f} t/ha')
print()
delta_v1_v3 = res_top30['RMSE_mean'] - res_rf['RMSE_mean']
if abs(delta_v1_v3) < 0.05:
    print('     ✅ Las diferencias entre v1, v2 y v3 son menores a 0.05 t/ha.')
    print('        Esto confirma que la señal predictiva está concentrada en')
    print('        pocas variables (principalmente radiación y temperatura) y')
    print('        que las features adicionales no aportan señal independiente.')
    print('        Con n~170 obs, un modelo parsimonioso (top-30) es')
    print('        estadísticamente más apropiado que uno con 70+ features.')
elif delta_v1_v3 < 0:
    print('     ✅ La reducción a top-30 MEJORÓ el RMSE_LOPO.')
    print('        Las features eliminadas eran ruido que añadía varianza.')
    print('        El modelo top-30 es más preciso Y más parsimonioso.')
else:
    print(f'     ⚠️  La reducción a top-30 empeoró RMSE_LOPO en {delta_v1_v3:.4f} t/ha.')
    print('        Las features eliminadas sí aportaban señal útil.')
    print('        Considerar usar top-40 o top-50 en MOD-10.')
print()
print('  5. CONCLUSIÓN SOBRE CÓDIGO DANE MUNICIPIO:')
print('     Su eliminación es correcta metodológicamente.')
print('     Incluirlo permitiría al modelo predecir por municipio')
print('     (efecto fijo geográfico) en lugar de por clima.')
print('     El objetivo científico del proyecto es cuantificar la')
print('     relación clima→rendimiento, no el rendimiento histórico')
print('     por municipio. Esta distinción es fundamental.')
print()
print('  6. LIMITACIÓN FUNDAMENTAL DEL DATASET:')
print('     Con n~170 obs, correlaciones máx r=0.44 (MOD-08) y')
print('     10 folds LOPO, ningún modelo alcanzará R²_LOPO alto.')
print('     Un RMSE_LOPO de ~1.3 t/ha sobre una media de ~6 t/ha')
print('     implica un error relativo del ~20%, que es el límite')
print('     de lo que los datos actuales pueden predecir.')
print('     Para reducirlo: ampliar el dataset a 5+ años adicionales')
print('     o incorporar datos de suelo y variedades cultivadas.')
print()
print(sep)
print(f'  ✅ MOD-09 COMPLETADO')
print(sep)


In [ ]:
# ============================================================
# RESUMEN FINAL — MOD-09
# ============================================================

sep = '='*65
print(sep)
print('  ✅ MOD-09 — Modelado Predictivo  COMPLETADO')
print(sep)
print()
print('📦 Archivos generados:')
archivos = [
    ('M1_resultados_modelos.xlsx',  '7 hojas: LOPO, folds, predicciones, importancia, RF_Top30, features, params'),
    ('M1_modelo_final.pkl',         f'Modelo final: RF_Top{TOP_N_FEATURES}_Regularizado'),
    ('M2_shap_values.pkl',          'SHAP values para interpretabilidad'),
    ('M1_Dashboard_Modelos.png',    'Dashboard 9 paneles — 4 modelos'),
    ('M1_RF_PCA_LOPO_Comparacion.png',   f'LOPO RF v1 vs RF+PCA ({N_COMPONENTES_PCA} comp.)'),
    ('M1_RF_PCA_Matrices.png',      'Matrices confusion evolucion RF (v1/v2/PCA)'),
    ('M1_RF_Top30_ROC.png',         'ROC RF v1 vs v3'),
    ('M1_RF_PCA_Dashboard.png',     'Dashboard PCA: varianza, codo, correlacion, cargas'),
    ('M1_RF_PCA_SHAP.png',          f'SHAP beeswarm sobre componentes PCA'),
    ('M1_RF_Optimizado.pkl',        'RF v2 regularizado'),
    ('M1_RF_Top30.pkl',             f'RF v3 top-{TOP_N_FEATURES} — MODELO FINAL'),
    ('M1_Matrices_Confusion.png',   'Matrices de confusión 4 modelos'),
    ('M1_Curvas_ROC.png',           'Curvas ROC multiclase 4 modelos'),
    ('M1_SHAP_Beeswarm.png',        f'SHAP beeswarm top-20 features ({mejor_nombre})'),
]
for archivo, desc in archivos:
    print(f'   📄 data/master/{archivo}')
    print(f'      {desc}')

print()
print('📊 Resultados LOPO (generalización fuera de muestra):')
print(f'   {"Modelo":<22}  {"RMSE":>8}  {"±std":>6}  {"R²":>7}  {"MAPE":>7}')
print(f'   {"-"*55}')
for nombre, resumen in zip(NAMES_MOD, [res_ridge,res_rf,res_xgb,res_lgb]):
    marca = ' ★' if nombre == mejor_nombre else ''
    print(f'   {nombre:<22}  {resumen["RMSE_mean"]:>8.4f}  '
          f'{resumen["RMSE_std"]:>6.4f}  {resumen["R2_mean"]:>7.4f}  '
          f'{resumen["MAPE_mean"]:>6.1f}%{marca}')

print()
print(f'  ★ Modelo seleccionado: RF Top-{TOP_N_FEATURES} (Regularizado)')
print()
print('  Evolución del modelo RF:')
print(f'    v1 (todas features)   : RMSE_LOPO={res_rf["RMSE_mean"]:.4f}±{res_rf["RMSE_std"]:.4f}')
print(f'    v2 (regularizado)     : RMSE_LOPO={res_rf_opt_full["RMSE_mean"]:.4f}±{res_rf_opt_full["RMSE_std"]:.4f}')
print(f'    v3 (PCA {N_COMPONENTES_PCA} comp.)   : RMSE_LOPO={res_top30["RMSE_mean"]:.4f}±{res_top30["RMSE_std"]:.4f}')
print()
print('➡️  Siguiente módulo : MOD-10 — Evaluación Final y Reportes')
print('    Entrada          : M1_modelo_final.pkl + M2_shap_values.pkl')
print(sep)
